# PC-FNO3D v7 — version commentée ligne par ligne

Même code que `fno3d_v7_1.ipynb` (vérifié identique), avec un commentaire en français au-dessus de chaque ligne.

**Note :** le BLOC 9b (raffinement) a été testé mais **non retenu** — seul le BLOC 9 a été utilisé pour le modèle final.

## BLOC 0 — Montage du Drive, imports, GPU

In [ ]:
# BLOC 0 — Montage du Drive, imports, GPU
# os : système (variables d'environnement) ; json : fichiers JSON ; math : sin, cos, pi ;
# time : chronomètre ; random : tirages aléatoires ; gc : libération de la mémoire
import os, json, math, time, random, gc
# Path : manipuler les chemins de fichiers (ex. RUN_DIR / "best.pt")
from pathlib import Path

# numpy : calcul sur des tableaux de nombres
import numpy as np
# pandas : tableaux de données (CSV, index des instants)
import pandas as pd
# torch : PyTorch, la bibliothèque de deep learning
import torch
# nn : les couches du réseau (Conv3d, Module...)
import torch.nn as nn
# F : fonctions sans paramètres (pad, gelu...)
import torch.nn.functional as F
# Dataset / DataLoader : produire et charger les exemples pendant l'entraînement
from torch.utils.data import Dataset, DataLoader
# gaussian_filter : lisser le relief ; binary_erosion : rétrécir un masque d'un pixel
from scipy.ndimage import gaussian_filter, binary_erosion
# pyarrow.csv : lecture très rapide des gros fichiers CSV
import pyarrow.csv as pacsv

# Mode test : vrai seulement si la variable d'environnement ANEO_TEST vaut "1" (mini-dataset local)
TEST_MODE = os.environ.get("ANEO_TEST") == "1"      # validation locale sur mini-dataset

# Sur Colab (pas en mode test) : on monte Google Drive pour lire les données et sauvegarder les résultats
if not TEST_MODE:
    # module Colab qui donne accès au Drive
    from google.colab import drive
    # le Drive apparaît dans /content/drive/MyDrive/...
    drive.mount("/content/drive")

# Choisit le GPU ("cuda") s'il est disponible, sinon le CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Affiche le matériel utilisé
print("Device :", device)
# Si on a un GPU :
if device.type == "cuda":
    # nom du GPU (ex. NVIDIA A100)
    print("GPU    :", torch.cuda.get_device_name(0))
    # mémoire du GPU en gigaoctets
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")
    # autorise le format TF32 pour les multiplications de matrices (plus rapide, perte négligeable)
    torch.backends.cuda.matmul.allow_tf32 = True
    # idem pour les convolutions
    torch.backends.cudnn.allow_tf32 = True

# Graine aléatoire fixe : mêmes tirages à chaque exécution (résultats reproductibles)
SEED = 42
# applique la graine à Python, NumPy et PyTorch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# message de fin du bloc
print("BLOC 0 OK")

## BLOC 1 — Configuration

In [ ]:
# BLOC 1 — Configuration
# Securite : si le BLOC 0 n'a pas ete execute (redemarrage de session), on recharge l'essentiel.
# globals() = variables existantes ; si TEST_MODE n'existe pas, le BLOC 0 n'a pas été lancé
if "TEST_MODE" not in globals():
    # on recharge les modules de base
    import os, json, math, time, random, gc
    # chemins de fichiers
    from pathlib import Path
    # calcul numérique
    import numpy as np
    # tableaux de données
    import pandas as pd
    # PyTorch
    import torch
    # mode test (mini-dataset) ou non
    TEST_MODE = os.environ.get("ANEO_TEST") == "1"
    # GPU si disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # avertit l'utilisateur de relancer le BLOC 0 (montage du Drive)
    print("BLOC 0 non execute : relance-le pour monter le Drive et charger tous les imports.")

# En mode test : les dossiers sont donnés par des variables d'environnement
if TEST_MODE:
    # dossier du mini-dataset
    DATA_DIR = Path(os.environ["ANEO_DATA"])
    # dossier du cache de tenseurs
    CACHE_DIR = Path(os.environ["ANEO_CACHE"])
    # dossier des résultats
    RUN_DIR = Path(os.environ["ANEO_RUN"])
# Sur Colab :
else:
    # dataset brut (csv.gz, meta.json, grille) sur le Drive
    DATA_DIR = Path("/content/drive/MyDrive/aneo_dataset_3d_v2")   # dataset genere v2 (csv.gz)
    # cache sur le disque local de Colab (lecture beaucoup plus rapide que le Drive)
    CACHE_DIR = Path("/content/aneo_cache_v7")                      # tenseurs .npy (disque local)
    # résultats (modèles, statistiques, figures) sur le Drive pour ne pas les perdre
    RUN_DIR = Path("/content/drive/MyDrive/aneo_fno3d_runs/v7")     # checkpoints, resultats, figures
# Pour chaque dossier nécessaire :
for d in (CACHE_DIR, RUN_DIR, RUN_DIR / "figures"):
    # le crée s'il n'existe pas (et ses parents), sans erreur s'il existe déjà
    d.mkdir(parents=True, exist_ok=True)

# Cache persistant sur le Drive (v7 : surface moyennee par blocs 2x2, incompatible avec le cache v6)
# shutil : copier des fichiers et des dossiers
import shutil
# copie du cache gardée sur le Drive (aucune en mode test)
CACHE_DRIVE = None if TEST_MODE else Path("/content/drive/MyDrive/aneo_cache_v7")

# Si une copie Drive complète existe et que le cache local est vide (nouvelle session Colab) :
if CACHE_DRIVE is not None and (CACHE_DRIVE / "index.csv").exists() and not (CACHE_DIR / "index.csv").exists():
    # message
    print("Copie du cache depuis le Drive vers le disque local...")
    # démarre le chronomètre
    t0 = time.time()
    # recopie tout le cache du Drive vers le disque local (évite de tout reconstruire)
    shutil.copytree(CACHE_DRIVE, CACHE_DIR, dirs_exist_ok=True)
    # durée de la copie en minutes
    print(f"Cache copie en {(time.time() - t0) / 60:.1f} min")

# Ouvre meta.json (description du dataset : grille, niveaux...)
with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    # META = dictionnaire Python des métadonnées
    META = json.load(f)
# Ouvre split_days.json (liste des jours train / val / test)
with open(DATA_DIR / "split_days.json", encoding="utf-8") as f:
    # SPLIT = dictionnaire des trois listes de jours
    SPLIT = json.load(f)

# Geometrie (lue depuis meta.json du generateur)
# taille de la grille des capteurs : 1000 colonnes x 500 lignes
NX_S, NY_S = META["grille_surface"]["nx"], META["grille_surface"]["ny"]
# pas de la grille des capteurs en x : 50 m
DX_S = float(META["grille_surface"]["dx_m"])
# pas en y : 50 m
DY_S = float(META["grille_surface"]["dy_m"])
# facteur entre grille capteurs et grille du modèle : 2
STRIDE = META["noeuds_3d"]["stride"]
# grille du modèle : 500 x 250
NX, NY = NX_S // STRIDE, NY_S // STRIDE                     # grille du modele (100 m)
# pas de la grille du modèle : 100 m
DX, DY = DX_S * STRIDE, DY_S * STRIDE
# les 12 altitudes (m au-dessus du niveau de la mer)
Z_LEVELS = np.array(META["noeuds_3d"]["z_levels_asl_m"], dtype=np.float64)
# nombre de niveaux : 12
NZ = len(Z_LEVELS)
# point_id à partir duquel les points sont des nœuds 3D (en dessous : capteurs de surface)
NODE_ID_START = META["noeuds_3d"]["point_id_debut"]
# lissage du relief (300 m) pour la condition au sol, comme le générateur
TERRAIN_SMOOTH_M = 300.0     # meme lissage que le generateur pour la condition au sol

# Modele
# nombre de canaux internes du réseau
WIDTH = 48
# fréquences de Fourier gardées en (z, y, x)
MODES = (6, 32, 32)          # (z, y, x) : plus de modes pour les structures fines (DSO3)
# nombre de couches de Fourier
N_LAYERS = 4
# ajoute la convolution locale 3x3x3 dans chaque couche
LOCAL_CONV = True            # branche convolutive locale 3x3x3 (details fins, DSO3)
# moyenne des capteurs par bloc 2x2 (agrégation)
POOL_SURFACE = True          # surface : moyenne des capteurs par bloc 2x2 au lieu d'1 capteur sur 4
# largeur de la tête de sortie
PROJ_HIDDEN = 128            # largeur de la tete de projection
# zéros ajoutés aux bords avant la FFT
PAD = (4, 8, 8)              # rembourrage (z, y, x) : limite la periodicite de la FFT

# Entrainement
# nombre d'époques
EPOCHS = 140
# instants tirés au hasard par époque
STEPS_PER_EPOCH = 400        # instants tires au hasard par epoque
# nombre d'instants de validation
VAL_SAMPLES = 64             # sous-ensemble fixe de validation
# un instant à la fois (1,5 million de points par instant)
BATCH_SIZE = 1
# gradients de 2 instants accumulés avant chaque mise à jour
ACCUM_STEPS = 2
# taux d'apprentissage maximal
LR = 1e-3
# petite pénalité sur les poids (contre le surapprentissage)
WEIGHT_DECAY = 1e-4
# la physique monte progressivement sur 10 époques
WARMUP_PHYS_EPOCHS = 10      # montee progressive des termes physiques
# 4 processus préparent les données en parallèle
NUM_WORKERS = 4
# coefficient de la moyenne mobile des poids
EMA_DECAY = 0.999             # moyenne exponentielle des poids (modele evalue et sauvegarde)

# Masquage des capteurs (entree)
# en entraînement : 20 à 100 % des capteurs restent visibles
KEEP_TRAIN = (0.2, 1.0)      # fraction de capteurs visibles, tiree a chaque echantillon
# en validation / test : 50 % visibles
KEEP_EVAL = 0.5              # fraction fixe en validation / test

# Poids de la loss
# poids du terme de masse
LAMBDA_MASS = 1.0            # applique a L_mass / MASS_REF
# échelle du terme de masse (on divise L_mass par cette valeur)
MASS_REF = 5e-7              # echelle du terme de masse dans la loss (DSO2 avec marge)
# poids de la condition au sol
LAMBDA_BC = 0.1
# terme de lissage désactivé
LAMBDA_SMOOTH = 0.0          # donnees deja lisses : terme desactive par defaut
# poids de la loss de gradient
LAMBDA_GRAD = 0.15           # loss sur les gradients horizontaux (corrige le lissage, DSO3 et w)
# poids de base de w
W_WEIGHT = 1.0               # poids de w dans L_data
# plafond du poids de w par niveau
W_LEVEL_MAX = 25.0           # plafond du poids de w par niveau (w est tres faible en altitude)
# objectif de R2 pour u, v et w
DSO_R2_ALL = 0.95            # objectif : R2 global >= 0.95 pour u, v ET w a chaque niveau

# Seuils des DSO
# DSO1 : RMSE maximal (m/s)
DSO1_RMSE = 0.5              # m/s, u et v, a chaque niveau
# DSO1 : R2 minimal
DSO1_R2 = 0.95               # R2 global par niveau
# DSO2 : résidu de masse maximal
DSO2_RMS = 1e-4              # kg.m-3.s-1, RMS du residu de continuite
# DSO3 : rapport de rugosité maximal 1,1
DSO3_RANGE = (0.0, 1.1)      # DSO3 : pas de bruit artificiel (rugosite <= 1.1 x verite)
# DSO4 : temps maximal par instant (s)
DSO4_TIME = 1.0              # s par instant
# DSO5 : RMSE maximal avec 25 % des capteurs
DSO5_RMSE = 0.7              # m/s avec 25 % des capteurs
# un niveau doit couvrir au moins 5 % du domaine pour entrer dans les DSO
MIN_LEVEL_COVER = 0.05       # niveaux couvrant moins de 5 % du domaine exclus des DSO par niveau
# paramètre de Coriolis à 36° N (vent géostrophique)
F_CORIOLIS = 8.57e-5         # s-1, latitude ~36 N

# En mode test : modèle minuscule et 2 époques (vérifier que le code tourne)
if TEST_MODE:
    # architecture réduite
    WIDTH, MODES, N_LAYERS, PAD, PROJ_HIDDEN = 4, (4, 8, 8), 2, (2, 4, 4), 4
    # entraînement très court
    EPOCHS, STEPS_PER_EPOCH, VAL_SAMPLES, NUM_WORKERS = 2, 3, 2, 0
    # physique dès la première époque
    WARMUP_PHYS_EPOCHS = 1

# Affiche la taille de la grille du modèle
print(f"Grille modele : NZ={NZ} x NY={NY} x NX={NX} (dx={DX:.0f} m)")
# Affiche les niveaux
print(f"Niveaux ASL   : {Z_LEVELS.tolist()}")
# Affiche le nombre de jours par ensemble
print(f"Split jours   : train={len(SPLIT['train_days'])} val={len(SPLIT['val_days'])} "
      f"test={len(SPLIT['test_days'])}")
# fin du bloc
print("BLOC 1 OK")

## BLOC 2 — Grille statique, masques, operateurs

In [ ]:
# BLOC 2 — Grille statique, masques, operateurs
# Lit le fichier de la grille : pour chaque point, son identifiant, sa position et ses propriétés du sol
grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv",
                   usecols=["point_id", "ix", "iy", "z_terrain_m", "z_rel_m", "z0_m",
                            "slope_deg", "aspect_deg"])

# Capteurs de surface (point_id < NODE_ID_START), triés ligne par ligne (iy puis ix)
surf = grid[grid["point_id"] < NODE_ID_START].sort_values(["iy", "ix"]).reset_index(drop=True)
# Vérifie qu'il y a bien 1000 x 500 = 500 000 capteurs
assert len(surf) == NX_S * NY_S, "Grille de surface incomplete"
# Nœuds 3D (point_id >= NODE_ID_START) : 849 184 points en altitude
nodes = grid[grid["point_id"] >= NODE_ID_START].reset_index(drop=True)

# Transforme une colonne de la surface (500 000 valeurs) en carte 2D (500 lignes x 1000 colonnes)
def s2d(col):
    # reshape en (lignes, colonnes), en float64 pour la précision
    return surf[col].values.reshape(NY_S, NX_S).astype(np.float64)

# Altitude du sol sur la grille de 50 m
ZT_50 = s2d("z_terrain_m")
# Altitude du sol sur la grille du modèle (un point sur deux -> 100 m)
zt = ZT_50[::STRIDE, ::STRIDE]
# Rugosité z0, au minimum 1e-4 (pour pouvoir en prendre le logarithme)
z0 = np.maximum(s2d("z0_m")[::STRIDE, ::STRIDE], 1e-4)
# Pente du terrain en degrés
slope = s2d("slope_deg")[::STRIDE, ::STRIDE]
# Orientation de la pente, convertie en radians
aspect = np.radians(s2d("aspect_deg")[::STRIDE, ::STRIDE])
# Hauteur des capteurs au-dessus du sol (20 à 30 m)
zrel_sensor = s2d("z_rel_m")[::STRIDE, ::STRIDE]

# point_id de surface -> indice plat (grille 50 m)
# Table de correspondance remplie de -1 (= point inconnu)
PID_TO_SURF = np.full(int(surf["point_id"].max()) + 1, -1, dtype=np.int64)
# Pour chaque point_id de surface : sa position (0 à 499 999) dans la carte
PID_TO_SURF[surf["point_id"].values] = np.arange(len(surf))

# noeud 3D -> indice plat (NZ, NY, NX)
# Altitude absolue de chaque nœud = altitude du sol + hauteur relative
z_abs_nodes = nodes["z_terrain_m"].values + nodes["z_rel_m"].values
# Niveau le plus proche de cette altitude (indice 0 à 11)
k_nodes = np.abs(z_abs_nodes[:, None] - Z_LEVELS[None, :]).argmin(axis=1)
# Ligne du nœud sur la grille de 100 m
j_nodes = nodes["iy"].values // STRIDE
# Colonne du nœud sur la grille de 100 m
i_nodes = nodes["ix"].values // STRIDE
# Table point_id du nœud -> position dans le volume (rempli de -1)
NODE_FLAT = np.full(int(nodes["point_id"].max()) - NODE_ID_START + 1, -1, dtype=np.int64)
# Position aplatie = (niveau x NY + ligne) x NX + colonne
NODE_FLAT[nodes["point_id"].values - NODE_ID_START] = (k_nodes * NY + j_nodes) * NX + i_nodes

# Masque VALID : False partout au départ (12 x 250 x 500 cases)
VALID = np.zeros(NZ * NY * NX, dtype=bool)
# True là où un nœud existe (au-dessus du sol)
VALID[NODE_FLAT[NODE_FLAT >= 0]] = True
# Remis en forme 3D (niveaux, lignes, colonnes)
VALID = VALID.reshape(NZ, NY, NX)

# Masque du residu de masse : les deux niveaux valides ET les voisins horizontaux utilises
# par les differences centrees valides (sinon le sol, rempli de zeros, fausse la divergence).
# Motif en croix : le point et ses 4 voisins
cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
# Érosion : garde seulement les points dont les 4 voisins sont aussi valides (à chaque niveau)
VALID_INTERIOR = np.stack([binary_erosion(VALID[k], structure=cross, border_value=1)
                           for k in range(NZ)])
# Résidu calculable entre deux niveaux si les deux sont intérieurs (11 demi-niveaux)
MASS_MASK = VALID_INTERIOR[1:] & VALID_INTERIOR[:-1]

# Premier noeud valide de chaque colonne (condition cinematique au sol)
# Copie du masque VALID
FIRST_VALID = VALID.copy()
# Garde un nœud seulement si le niveau juste en dessous n'existe pas (= premier au-dessus du sol)
FIRST_VALID[1:] &= ~VALID[:-1]

# Pente du relief vu par l'ecoulement (meme lissage que le generateur)
# Relief lissé sur 300 m (sigma en nombre de cases de 50 m), puis ramené à 100 m
zt_flow = gaussian_filter(ZT_50, sigma=TERRAIN_SMOOTH_M / DX_S, mode="nearest")[::STRIDE, ::STRIDE]
# Pentes du relief lissé : dz/dy et dz/dx (pour w = u.dz/dx + v.dz/dy)
DZT_DY, DZT_DX = np.gradient(zt_flow, DY, DX)

# Champs statiques d'entree (normalises a la main, ordres de grandeur O(1))
# Altitude de chaque niveau recopiée sur toute la grille (12 x 250 x 500)
z_asl = np.broadcast_to(Z_LEVELS[:, None, None], (NZ, NY, NX))
# Hauteur au-dessus du sol (0 si sous le sol)
z_agl = np.clip(z_asl - zt[None], 0.0, None)
# Logarithme de la rugosité
logz0 = np.log(z0)
# Les 6 canaux de relief, chacun ramené à l'ordre de grandeur 1
STATIC_2D = np.stack([
    # altitude du sol / 500
    zt / 500.0,
    # log z0 standardisé (moyenne 0, écart-type 1)
    (logz0 - logz0.mean()) / (logz0.std() + 1e-6),
    # pente / 10
    slope / 10.0,
    # orientation codée sur un cercle : sinus
    np.sin(aspect),
    # et cosinus (359° et 1° restent voisins)
    np.cos(aspect),
    # hauteur du capteur centrée sur 25 m
    (zrel_sensor - 25.0) / 5.0,
]).astype(np.float32)
# Les 3 canaux de géométrie par niveau : altitude / 1000, hauteur au-dessus du sol / 1000, masque VALID
STATIC_3D = np.stack([z_asl / 1000.0, z_agl / 1000.0, VALID.astype(np.float64)]).astype(np.float32)

# Nombre de variables de surface : u, v, w, T, rho, HR, p
N_SURF = 7                                   # u, v, w, T, rho, RH, p
# Coordonnees centrees et altitude des capteurs (estimation du vent geostrophique)
# Coordonnée x de chaque case, centrée sur le milieu du domaine (en m)
XC = (np.arange(NX) * DX)[None, :].repeat(NY, 0) - (NX - 1) * DX / 2
# Coordonnée y centrée (en m)
YC = (np.arange(NY) * DY)[:, None].repeat(NX, 1) - (NY - 1) * DY / 2
# Altitude absolue moyenne des 4 capteurs de chaque bloc 2x2 (sinon celle du capteur gardé)
Z_SENSOR = (ZT_50 + s2d("z_rel_m")).reshape(NY, STRIDE, NX, STRIDE).mean(axis=(1, 3)) if POOL_SURFACE \
    else zt + zrel_sensor

# Agrégation 2x2 des capteurs (masked average pooling)
def pool_surface(S):
    """(8, NY_S*NX_S ou NY_S, NX_S) -> (8, NY, NX). Canaux 0-6 : variables, canal 7 : capteur disponible.
    POOL_SURFACE : moyenne des capteurs disponibles de chaque bloc STRIDE x STRIDE (tous les capteurs
    sont utilises et le bruit de mesure est reduit) ; sinon un capteur sur STRIDE**2."""
    # Remet S en cartes 2D : 8 canaux x 500 x 1000
    S = S.reshape(8, NY_S, NX_S)
    # Sans agrégation : on garde un capteur sur 4
    if not POOL_SURFACE:
        # un point sur deux en ligne et en colonne
        return S[:, ::STRIDE, ::STRIDE].copy()
    # Masque des capteurs disponibles (canal 7 > 0,5), regroupé en blocs de 2x2
    m = (S[7] > 0.5).astype(np.float32).reshape(NY, STRIDE, NX, STRIDE)
    # Nombre de capteurs disponibles dans chaque bloc (0 à 4)
    cnt = m.sum(axis=(1, 3))
    # Résultat vide : 8 canaux sur la grille de 100 m
    out = np.zeros((8, NY, NX), dtype=np.float32)
    # Pour chacune des 7 variables :
    for c in range(7):
        # valeurs regroupées en blocs 2x2
        v = S[c].reshape(NY, STRIDE, NX, STRIDE)
        # moyenne des capteurs disponibles du bloc ; 0 si aucun capteur
        out[c] = np.where(cnt > 0, (v * m).sum(axis=(1, 3)) / np.maximum(cnt, 1.0), 0.0)
    # Canal 7 : 1 si au moins un capteur disponible dans le bloc
    out[7] = (cnt > 0).astype(np.float32)
    # Renvoie la surface agrégée (8, 250, 500)
    return out

# 8 variables temps / physique : heure (2), jour de l'année (2), vent géostrophique (2), vent thermique (2)
N_TFEAT = 8                    # heure (2), jour de l'annee (2), vent geostrophique (2), vent thermique (2)
# Nombre de canaux d'entrée : 7 + 1 + 6 + 3 + 8 = 25
C_IN = N_SURF + 1 + STATIC_2D.shape[0] + STATIC_3D.shape[0] + N_TFEAT
# Affiche le nombre de nœuds valides, de points du masque de masse et de canaux
print(f"Noeuds valides : {VALID.sum():,} / {VALID.size:,} "
      f"| masse : {MASS_MASK.sum():,} demi-niveaux | canaux d'entree : {C_IN}")

# Operateurs discrets identiques au generateur (np.gradient + demi-niveaux)
# Écarts d'altitude entre niveaux successifs (delta z), sur le GPU
DZ_T = torch.tensor(np.diff(Z_LEVELS), dtype=torch.float32, device=device)

# Dérivée numérique de f le long de la dimension dim, avec un pas h
def grad1d(f, h, dim):
    """Differences centrees a l'interieur, decentrees aux bords (equivalent np.gradient)."""
    # nombre de points dans cette direction
    n = f.shape[dim]
    # premier bord : (f[1] - f[0]) / h
    first = (f.narrow(dim, 1, 1) - f.narrow(dim, 0, 1)) / h
    # intérieur : (f[i+1] - f[i-1]) / (2h), différence centrée
    inner = (f.narrow(dim, 2, n - 2) - f.narrow(dim, 0, n - 2)) / (2.0 * h)
    # dernier bord : (f[n-1] - f[n-2]) / h
    last = (f.narrow(dim, n - 1, 1) - f.narrow(dim, n - 2, 1)) / h
    # recolle les trois morceaux
    return torch.cat([first, inner, last], dim=dim)

# Résidu de l'équation de continuité (conservation de la masse)
def mass_residual(rho, u, v, w):
    """r(k+1/2) = (rho w[k+1] - rho w[k]) / dz + 0.5 (D[k] + D[k+1]) ; tenseurs (B, NZ, NY, NX)."""
    # divergence horizontale D = d(rho u)/dx + d(rho v)/dy à chaque niveau
    d = grad1d(rho * u, DX, -1) + grad1d(rho * v, DY, -2)
    # flux vertical de masse rho * w
    rw = rho * w
    # flux vertical entre deux niveaux / dz + moyenne des divergences des deux niveaux
    return (rw[:, 1:] - rw[:, :-1]) / DZ_T[None, :, None, None] + 0.5 * (d[:, 1:] + d[:, :-1])

# Masque VALID sur le GPU
VALID_T = torch.from_numpy(VALID).to(device)
# Paires de voisins en x toutes deux valides
GRAD_MASK_X_T = torch.from_numpy(VALID[:, :, 1:] & VALID[:, :, :-1]).to(device)
# Paires de voisins en y toutes deux valides
GRAD_MASK_Y_T = torch.from_numpy(VALID[:, 1:, :] & VALID[:, :-1, :]).to(device)
# Masque du résidu de masse sur le GPU
MASS_MASK_T = torch.from_numpy(MASS_MASK).to(device)
# Premier nœud au-dessus du sol, sur le GPU
FIRST_VALID_T = torch.from_numpy(FIRST_VALID).to(device)
# Pente dz/dx sur le GPU
DZT_DX_T = torch.tensor(DZT_DX, dtype=torch.float32, device=device)
# Pente dz/dy sur le GPU
DZT_DY_T = torch.tensor(DZT_DY, dtype=torch.float32, device=device)
# 6 canaux de relief sur le GPU
STATIC_2D_T = torch.from_numpy(STATIC_2D).to(device)
# 3 canaux de géométrie sur le GPU
STATIC_3D_T = torch.from_numpy(STATIC_3D).to(device)
# Supprime le gros tableau de la grille (1,35 million de lignes)
del grid
# Libère la mémoire
gc.collect()
# fin du bloc
print("BLOC 2 OK")

## BLOC 3 — Construction du cache de tenseurs (une seule fois)

In [ ]:
# BLOC 3 — Construction du cache de tenseurs (une seule fois)
# Pour chaque instant :
#   <jour>_<HHMMSS>_S.npy : surface (8, NY, NX) = u, v, w, T, rho, RH, p, capteur_disponible
#   <jour>_<HHMMSS>_Y.npy : volume (4, NZ, NY, NX) = u, v, w, rho (0 sous le terrain)
# Colonnes lues dans les CSV (les autres sont ignorées pour aller plus vite)
READ_COLS = ["point_id", "timestamp_utc", "temperature_c", "air_density_kg_m3",
             "relative_humidity_pct", "pressure_pa", "wind_u_m_s", "wind_v_m_s",
             "wind_w_m_s", "mask_i_obs"]
# Les 7 variables de surface, dans l'ordre des canaux de S
SURF_VARS = ["wind_u_m_s", "wind_v_m_s", "wind_w_m_s", "temperature_c",
             "air_density_kg_m3", "relative_humidity_pct", "pressure_pa"]

# Numéro du jour d'après le nom du fichier (aneo_observations_day_004_... -> 4)
def day_of(path):
    # 4e morceau du nom découpé par "_"
    return int(path.name.split("_")[3])

# Ensemble (train / val / test) d'un jour
def split_of(day):
    # jour d'entraînement
    if day in SPLIT["train_days"]:
        return "train"
    # jour de validation
    if day in SPLIT["val_days"]:
        return "val"
    # sinon jour de test
    return "test"

# Transforme le fichier CSV d'un jour en fichiers S et Y pour chacun de ses instants
def build_day(path):
    # numéro du jour
    day = day_of(path)
    # options de lecture : seulement nos colonnes, horodatage lu comme texte
    opts = pacsv.ConvertOptions(include_columns=READ_COLS,
                                column_types={"timestamp_utc": "string"})
    # lecture rapide du CSV avec pyarrow
    tbl = pacsv.read_csv(path, convert_options=opts)
    # identifiants des points
    pid = tbl.column("point_id").to_numpy()
    # horodatages
    ts = tbl.column("timestamp_utc").to_pandas()
    # numéro de chaque instant distinct (codes) et liste des instants (uniq), triés
    codes, uniq = pd.factorize(ts, sort=True)
    # variables numériques converties en float32 (moins de mémoire)
    cols = {c: tbl.column(c).to_numpy().astype(np.float32) for c in READ_COLS[2:]}
    # libère la table pyarrow
    del tbl
    # lignes de l'index à remplir
    rows = []
    # Pour chaque instant c du jour :
    for c, stamp in enumerate(uniq):
        # indices des lignes de cet instant
        sel = np.nonzero(codes == c)[0]
        # leurs point_id
        p = pid[sel]
        # vrai pour les capteurs de surface
        is_s = p < NODE_ID_START
        # point_id de surface (ps) et des nœuds 3D (pn)
        ps, pn = p[is_s], p[~is_s]
        # lignes de surface (sel_s) et de nœuds (sel_n)
        sel_s, sel_n = sel[is_s], sel[~is_s]

        # Surface vide : 8 canaux x 500 000 capteurs
        S = np.zeros((8, NY_S * NX_S), dtype=np.float32)
        # position de chaque capteur dans la carte
        flat = PID_TO_SURF[ps]
        # Pour chacune des 7 variables :
        for ci, var in enumerate(SURF_VARS):
            # place les mesures à leur position
            S[ci, flat] = cols[var][sel_s]
        # Canal 8 : 1 = mesure réelle, 0 = valeur interpolée
        S[7, flat] = cols["mask_i_obs"][sel_s]
        # Agrégation 2x2 -> (8, 250, 500)
        S = pool_surface(S)

        # Volume cible vide : 4 variables x 1,5 million de positions (0 sous le relief)
        Y = np.zeros((4, NZ * NY * NX), dtype=np.float32)
        # position de chaque nœud 3D dans le volume
        nflat = NODE_FLAT[pn - NODE_ID_START]
        # Pour u, v, w et la densité :
        for ci, var in enumerate(["wind_u_m_s", "wind_v_m_s", "wind_w_m_s", "air_density_kg_m3"]):
            # place les valeurs vraies à leur position
            Y[ci, nflat] = cols[var][sel_n]
        # Remise en forme (4, 12, 250, 500)
        Y = Y.reshape(4, NZ, NY, NX)

        # horodatage de l'instant
        tstamp = pd.Timestamp(stamp)
        # nom unique de l'instant : jour_HHMMSS (ex. 004_120000)
        stem = f"{day:03d}_{tstamp.strftime('%H%M%S')}"
        # sauvegarde de la surface
        np.save(CACHE_DIR / f"{stem}_S.npy", S)
        # sauvegarde de la cible
        np.save(CACHE_DIR / f"{stem}_Y.npy", Y)
        # ligne d'index : nom, jour, horodatage, ensemble, heure décimale (minutes et secondes), jour de l'année
        rows.append({"stem": stem, "day": day, "timestamp": str(tstamp), "split": split_of(day),
                     "hour": tstamp.hour + tstamp.minute / 60 + tstamp.second / 3600,
                     "doy": tstamp.dayofyear})
    # renvoie les lignes d'index du jour
    return rows

# Chemin de l'index des instants
index_path = CACHE_DIR / "index.csv"
# Jours déjà traités (reprise si Colab s'est arrêté)
done_days = set(pd.read_csv(index_path)["day"]) if index_path.exists() else set()
# Liste des fichiers journaliers du dataset
day_files = sorted(DATA_DIR.glob("aneo_observations_day_*_multitime.csv*"))
# Ignore les téléchargements incomplets (.part)
day_files = [p for p in day_files if not p.name.endswith(".part")]
# Jours restant à traiter
todo = [p for p in day_files if day_of(p) not in done_days]
# Affiche l'avancement
print(f"Jours disponibles : {len(day_files)} | deja en cache : {len(done_days)} | a traiter : {len(todo)}")

# Chronomètre
t0 = time.time()
# Pour chaque jour restant :
for n, path in enumerate(todo, 1):
    # construit S et Y de tous ses instants
    rows = build_day(path)
    # transforme ses lignes d'index en tableau
    df_new = pd.DataFrame(rows)
    # les ajoute à index.csv (en-tête seulement si le fichier n'existe pas)
    df_new.to_csv(index_path, mode="a", header=not index_path.exists(), index=False)
    # temps écoulé
    el = time.time() - t0
    # affiche le jour traité et le temps restant estimé
    print(f"  jour {day_of(path):03d} : {len(rows)} instants | {n}/{len(todo)} "
          f"| reste ~{el / n * (len(todo) - n) / 60:.0f} min")
    # libère la mémoire
    gc.collect()

# Index final : sans doublon, trié par nom d'instant
INDEX = pd.read_csv(index_path).drop_duplicates("stem").sort_values("stem").reset_index(drop=True)
# Nombre d'instants par ensemble
print(INDEX["split"].value_counts().to_dict())

# Sauvegarde du cache sur le Drive (uniquement les fichiers absents)
if CACHE_DRIVE is not None:
    # crée le dossier sur le Drive
    CACHE_DRIVE.mkdir(parents=True, exist_ok=True)
    # fichiers déjà présents sur le Drive
    existing = {p_.name for p_ in CACHE_DRIVE.glob("*.npy")}
    # fichiers locaux pas encore copiés
    new_files = [p_ for p_ in sorted(CACHE_DIR.glob("*.npy")) if p_.name not in existing]
    # s'il y en a :
    if new_files:
        # message
        print(f"Sauvegarde du cache sur le Drive : {len(new_files)} fichiers...")
        # copie chaque nouveau fichier
        for p_ in new_files:
            shutil.copy2(p_, CACHE_DRIVE / p_.name)
    # copie aussi l'index
    shutil.copy2(CACHE_DIR / "index.csv", CACHE_DRIVE / "index.csv")
# fin du bloc
print("BLOC 3 OK")

## BLOC 4 — Controle de la verite terrain

In [ ]:
# BLOC 4 — Controle de la verite terrain
# Le residu de masse et la condition au sol doivent etre ~0 sur les donnees : sinon la loss
# physique contredirait les donnees et DSO2 serait inatteignable.
# Charge le volume cible Y d'un instant sur le GPU
def load_y(stem):
    return torch.from_numpy(np.load(CACHE_DIR / f"{stem}_Y.npy")).to(device)

# Résultats du contrôle
checks = []
# 5 instants tirés au hasard (toujours les mêmes grâce à random_state=0)
for stem in INDEX["stem"].sample(min(5, len(INDEX)), random_state=0):
    # cible de l'instant, avec une dimension batch ajoutée
    y = load_y(stem)[None]
    # sépare u, v, w et la densité
    u, v, w, rho = y[:, 0], y[:, 1], y[:, 2], y[:, 3]
    # résidu de continuité de la VÉRITÉ
    r = mass_residual(rho, u, v, w)
    # moyenne du résidu au carré sur les points intérieurs
    l_mass = (r[:, MASS_MASK_T] ** 2).mean().item()
    # w attendu au sol : vent horizontal x pente du terrain
    w_bc = u * DZT_DX_T + v * DZT_DY_T
    # écart au carré entre w vrai et w attendu, au premier nœud au-dessus du sol
    l_bc = ((w - w_bc)[:, FIRST_VALID_T] ** 2).mean().item()
    # mémorise les résultats
    checks.append((stem, l_mass, l_bc))
    # affiche les deux valeurs (doivent être quasi nulles)
    print(f"  {stem} : L_mass(verite) = {l_mass:.2e} | ecart condition sol = {l_bc:.2e} (m/s)^2")
# Arrête le notebook si la vérité ne respecte pas la conservation de la masse
assert max(c[1] for c in checks) < 1e-6, "La verite ne respecte pas la continuite : verifier le dataset"
# fin du bloc
print("BLOC 4 OK")

## BLOC 5 — Normalisation (jours d'entrainement uniquement)

In [ ]:
# BLOC 5 — Normalisation (jours d'entrainement uniquement)
# Fichier des statistiques de normalisation
stats_path = RUN_DIR / "norm_stats.json"
# S'il existe déjà, on le relit (les statistiques ne changent jamais d'une session à l'autre)
if stats_path.exists():
    # ouvre le fichier
    with open(stats_path, encoding="utf-8") as f:
        # NORM = dictionnaire des moyennes et écarts-types
        NORM = json.load(f)
# Sinon on les calcule :
else:
    # instants d'ENTRAÎNEMENT seulement (pas de fuite vers le test)
    tr = INDEX[INDEX["split"] == "train"]["stem"]
    # 200 instants au hasard suffisent
    tr = tr.sample(min(200, len(tr)), random_state=0)
    # sommes pour la surface : valeurs, carrés, nombre de capteurs
    s_sum = np.zeros(N_SURF); s_sq = np.zeros(N_SURF); s_n = 0
    # sommes pour la cible (u, v, w)
    y_sum = np.zeros(3); y_sq = np.zeros(3); y_n = 0
    # Pour chaque instant :
    for stem in tr:
        # surface en float64
        S = np.load(CACHE_DIR / f"{stem}_S.npy").astype(np.float64)
        # cases avec un capteur réel
        m = S[7] > 0.5
        # ajoute valeurs, carrés et nombre, seulement là où un capteur existe
        s_sum += S[:N_SURF, m].sum(axis=1); s_sq += (S[:N_SURF, m] ** 2).sum(axis=1); s_n += m.sum()
        # cible en float64
        Y = np.load(CACHE_DIR / f"{stem}_Y.npy").astype(np.float64)
        # idem pour u, v, w, seulement sur les nœuds valides (pas les zéros sous le relief)
        y_sum += Y[:3, VALID].sum(axis=1); y_sq += (Y[:3, VALID] ** 2).sum(axis=1); y_n += VALID.sum()
    # moyenne et écart-type de la surface : sd = racine(moyenne des carrés - moyenne^2)
    s_mu = s_sum / s_n; s_sd = np.sqrt(np.maximum(s_sq / s_n - s_mu ** 2, 1e-12))
    # moyenne et écart-type de la cible
    y_mu = y_sum / y_n; y_sd = np.sqrt(np.maximum(y_sq / y_n - y_mu ** 2, 1e-12))
    # rassemble les statistiques
    NORM = {"surf_mu": s_mu.tolist(), "surf_sd": s_sd.tolist(),
            "y_mu": y_mu.tolist(), "y_sd": y_sd.tolist(), "surf_vars": SURF_VARS}
    # les sauvegarde (le déploiement en a aussi besoin)
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(NORM, f, indent=2)

# Écart-type de w niveau par niveau (s'il n'est pas encore calculé)
if "w_level_sd" not in NORM:
    # instants d'entraînement
    tr = INDEX[INDEX["split"] == "train"]["stem"]
    # 200 au hasard
    tr = tr.sample(min(200, len(tr)), random_state=0)
    # sommes par niveau : valeurs, carrés, nombre
    wl_sum = np.zeros(NZ); wl_sq = np.zeros(NZ); wl_n = np.zeros(NZ)
    # Pour chaque instant :
    for stem in tr:
        # w uniquement (canal 2 de Y)
        Yw = np.load(CACHE_DIR / f"{stem}_Y.npy")[2].astype(np.float64)
        # pour chaque niveau :
        for k in range(NZ):
            # valeurs de w aux nœuds valides de ce niveau
            vals = Yw[k][VALID[k]]
            # ajoute aux sommes
            wl_sum[k] += vals.sum(); wl_sq[k] += (vals ** 2).sum(); wl_n[k] += vals.size
    # évite une division par zéro
    wl_n = np.maximum(wl_n, 1)
    # écart-type de w à chaque niveau
    NORM["w_level_sd"] = np.sqrt(np.maximum(wl_sq / wl_n - (wl_sum / wl_n) ** 2, 1e-12)).tolist()
    # met à jour le fichier
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(NORM, f, indent=2)

# Moyennes de surface sur le GPU, forme (7, 1, 1) pour s'appliquer à toute une carte
SURF_MU = torch.tensor(NORM["surf_mu"], dtype=torch.float32, device=device)[:, None, None]
# Écarts-types de surface
SURF_SD = torch.tensor(NORM["surf_sd"], dtype=torch.float32, device=device)[:, None, None]
# Moyennes de u, v, w
Y_MU = torch.tensor(NORM["y_mu"], dtype=torch.float32, device=device)
# Écarts-types de u, v, w
Y_SD = torch.tensor(NORM["y_sd"], dtype=torch.float32, device=device)
# poids de w par niveau : chaque niveau compte selon sa propre variabilite (sinon w en altitude,
# tres faible, est neglige par la loss et son R2 chute)
# Écart-type de w par niveau
W_LEVEL_SD = torch.tensor(NORM["w_level_sd"], dtype=torch.float32, device=device)
# Poids de w par niveau = (sd global / sd du niveau)^2, borné entre 1 et 25
W_LEVEL_W = torch.clamp((Y_SD[2] / W_LEVEL_SD) ** 2, 1.0, W_LEVEL_MAX)
# Affiche les poids de w
print("Poids de w par niveau :", np.round(W_LEVEL_W.cpu().numpy(), 2))
# Affiche moyennes et écarts-types de la cible
print("Cible  mu :", np.round(NORM["y_mu"], 3), " sd :", np.round(NORM["y_sd"], 3))
# fin du bloc
print("BLOC 5 OK")

## BLOC 6 — Dataset, DataLoaders, assemblage de l'entree

In [ ]:
# BLOC 6 — Dataset, DataLoaders, assemblage de l'entree
# Vent géostrophique estimé à partir de la pression des capteurs visibles
def geostrophic_from_surface(S, sensor):
    """
    Vent geostrophique estime a partir des capteurs visibles :
    pression reduite au niveau de la mer, plan ajuste par moindres carres, equilibre geostrophique
    ug = -(1 / rho f) dp/dy ; vg = (1 / rho f) dp/dx.
    """
    # carte vrai/faux des capteurs visibles
    m = sensor
    # moins de 100 capteurs : estimation peu fiable, on renvoie 0
    if m.sum() < 100:
        return 0.0, 0.0
    # pression des capteurs visibles (canal 6)
    p = S[6][m].astype(np.float64)
    # température en kelvin (canal 3 + 273,15)
    t_k = S[3][m].astype(np.float64) + 273.15
    # altitude des capteurs
    z = Z_SENSOR[m]
    # pression ramenée au niveau de la mer (formule barométrique) : supprime l'effet de l'altitude
    p_sl = p * np.exp(9.80665 * z / (287.05 * (t_k + 0.5 * 0.0065 * z)))
    # matrice du plan p = a + b.x + c.y
    A = np.stack([np.ones(m.sum()), XC[m], YC[m]], axis=1)
    # ajustement par moindres carrés : coef = [a, dp/dx, dp/dy]
    coef, *_ = np.linalg.lstsq(A, p_sl, rcond=None)
    # densité moyenne de l'air (canal 4)
    rho = float(S[4][m].mean())
    # équilibre géostrophique : ug = -(1/rho f) dp/dy, vg = (1/rho f) dp/dx
    return -coef[2] / (rho * F_CORIOLIS), coef[1] / (rho * F_CORIOLIS)

# Vent thermique estimé à partir de la température des capteurs visibles
def thermal_wind_from_surface(S, sensor):
    """
    Vent thermique (variation du vent geostrophique avec l'altitude, en s-1) estime a partir
    du gradient horizontal de temperature des capteurs visibles, ramenee au niveau de la mer :
    d(ug, vg)/dz = (g / (f T)) * (-dT/dy, dT/dx).
    """
    # capteurs visibles
    m = sensor
    # trop peu de capteurs : 0
    if m.sum() < 100:
        return 0.0, 0.0
    # température ramenée au niveau de la mer (+ 6,5 °C par km d'altitude)
    t_red = S[3][m].astype(np.float64) + 0.0065 * Z_SENSOR[m]
    # matrice du plan T = a + b.x + c.y
    A = np.stack([np.ones(m.sum()), XC[m], YC[m]], axis=1)
    # coef = [a, dT/dx, dT/dy]
    coef, *_ = np.linalg.lstsq(A, t_red, rcond=None)
    # facteur g / (f T), T en kelvin
    k = 9.80665 / (F_CORIOLIS * (float(t_red.mean()) + 273.15))
    # vent thermique : (g / f T) (-dT/dy, dT/dx)
    return -k * coef[2], k * coef[1]

# Les 8 variables temps / physique d'un instant
def time_geo_features(S, sensor, hour, doy):
    # vent géostrophique
    ug, vg = geostrophic_from_surface(S, sensor)
    # vent thermique
    su, sv = thermal_wind_from_surface(S, sensor)
    # sin/cos de l'heure, sin/cos du jour de l'année, puis vents ramenés à l'ordre de grandeur 1
    return np.array([math.sin(2 * math.pi * hour / 24), math.cos(2 * math.pi * hour / 24),
                     math.sin(2 * math.pi * doy / 365), math.cos(2 * math.pi * doy / 365),
                     ug / 10.0, vg / 10.0, su * 200.0, sv * 200.0], dtype=np.float32)

# Dataset PyTorch : produit un exemple (un instant) à partir d'un numéro
class ANEO3D(Dataset):
    """Renvoie la surface brute, les caracteristiques temporelles, la cible et le masque capteurs."""

    # Initialisation
    def __init__(self, stems, hours, doys, train, n_samples=None, seed=0):
        # noms des instants, heures décimales, jours de l'année
        self.stems, self.hours, self.doys = list(stems), list(hours), list(doys)
        # mode entraînement (tirages aléatoires) ou évaluation (fixe)
        self.train = train
        # nombre d'exemples par époque (entraînement)
        self.n_samples = n_samples
        # graine pour l'évaluation
        self.seed = seed

    # Nombre d'exemples
    def __len__(self):
        # 400 en entraînement, sinon le nombre d'instants
        return self.n_samples if self.n_samples else len(self.stems)

    # Produit l'exemple numéro i
    def __getitem__(self, i):
        # Entraînement :
        if self.train:
            # instant tiré au hasard
            j = np.random.randint(len(self.stems))
            # générateur aléatoire libre
            rng = np.random.default_rng()
            # part de capteurs visibles tirée entre 20 et 100 %
            keep = rng.uniform(*KEEP_TRAIN)
        # Évaluation :
        else:
            # instant numéro i
            j = i
            # graine fixe : mêmes capteurs masqués à chaque évaluation
            rng = np.random.default_rng(self.seed + i)
            # 50 % des capteurs visibles
            keep = KEEP_EVAL
        # nom de l'instant
        stem = self.stems[j]
        # charge la surface
        S = np.load(CACHE_DIR / f"{stem}_S.npy")
        # charge la cible
        Y = np.load(CACHE_DIR / f"{stem}_Y.npy")
        # masquage : visible si un capteur réel existe ET si le tirage aléatoire est < keep
        sensor = (S[7] > 0.5) & (rng.random(S[7].shape) < keep)
        # 8 variables temps / physique, calculées avec les seuls capteurs visibles
        tfeat = time_geo_features(S, sensor, self.hours[j], self.doys[j])
        # renvoie : 7 variables de surface, capteurs visibles, 8 variables, cible, numéro
        return (torch.from_numpy(S[:N_SURF]), torch.from_numpy(sensor.astype(np.float32)),
                torch.from_numpy(tfeat), torch.from_numpy(Y), j)

# Construit les 25 canaux d'entrée sur le GPU
def assemble_input(s_raw, sensor, tfeat):
    """Construit l'entree (B, C_IN, NZ, NY, NX) sur le GPU."""
    # taille du batch
    B = s_raw.shape[0]
    # normalise les 7 variables (z-score) et met 0 là où le capteur est masqué
    s = (s_raw - SURF_MU) / SURF_SD * sensor[:, None]
    # 7 mesures + 1 carte de disponibilité + 6 canaux de relief = 14 cartes 2D
    s2d = torch.cat([s, sensor[:, None], STATIC_2D_T.expand(B, -1, -1, -1)], dim=1)
    # recopie ces 14 cartes sur les 12 niveaux
    x2d = s2d[:, :, None].expand(-1, -1, NZ, -1, -1)
    # 3 canaux de géométrie par niveau
    x3d = STATIC_3D_T.expand(B, -1, -1, -1, -1)
    # les 8 variables temps / physique recopiées en chaque point du volume
    xt = tfeat[:, :, None, None, None].expand(-1, -1, NZ, NY, NX)
    # assemble : 14 + 3 + 8 = 25 canaux
    return torch.cat([x2d, x3d, xt], dim=1)

# Instants d'un ensemble (train / val / test)
def subset(split):
    # lignes de l'index de cet ensemble
    d = INDEX[INDEX["split"] == split]
    # noms, heures, jours de l'année
    return d["stem"].values, d["hour"].values, d["doy"].values

# instants d'entraînement
tr_s, tr_h, tr_d = subset("train")
# instants de validation
va_s, va_h, va_d = subset("val")
# instants de test
te_s, te_h, te_d = subset("test")
# Validation : instants synoptiques (toutes les 3 h) repartis regulierement sur les jours de validation
# horodatages des instants de validation
_ts_val = INDEX.set_index("stem").loc[va_s, "timestamp"]
# vrai pour les instants synoptiques (heure pleine, multiple de 3)
_is_syn = np.array([(lambda t: t.minute == 0 and t.second == 0 and t.hour % 3 == 0)(pd.Timestamp(x))
                    for x in _ts_val])
# candidats : les synoptiques (ou tous s'il n'y en a pas)
_cand = np.nonzero(_is_syn)[0] if _is_syn.any() else np.arange(len(va_s))
# 64 instants répartis régulièrement parmi les candidats
vsel = _cand[np.unique(np.linspace(0, len(_cand) - 1, min(VAL_SAMPLES, len(_cand))).astype(int))]

# Dataset d'entraînement : 400 exemples aléatoires par époque
train_ds = ANEO3D(tr_s, tr_h, tr_d, train=True, n_samples=STEPS_PER_EPOCH * BATCH_SIZE)
# Dataset de validation (graine 1000)
val_ds = ANEO3D(va_s[vsel], va_h[vsel], va_d[vsel], train=False, seed=1000)
# Dataset de test (graine 2000)
test_ds = ANEO3D(te_s, te_h, te_d, train=False, seed=2000)

# options communes : 4 processus de chargement, mémoire « épinglée » pour un transfert GPU rapide
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                 persistent_workers=NUM_WORKERS > 0)
# chargeur d'entraînement (pas de mélange : le tirage est déjà aléatoire)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
# chargeur de validation
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, **loader_kw)
# chargeur de test
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, **loader_kw)
# Affiche le nombre d'instants de chaque ensemble
print(f"Instants : train={len(tr_s)} | val={len(vsel)} (sur {len(va_s)}) | test={len(te_s)}")
# fin du bloc
print("BLOC 6 OK")

## BLOC 7 — Modele PC-FNO3D

In [ ]:
# BLOC 7 — Modele PC-FNO3D
# Couche de Fourier 3D
class SpectralConv3d(nn.Module):
    """Couche de Fourier 3D : poids complexes appris sur les modes conserves (z, y, x)."""

    # cin / cout : canaux d'entrée / sortie ; modes : fréquences gardées (6, 32, 32)
    def __init__(self, cin, cout, modes):
        # initialise la classe parente nn.Module
        super().__init__()
        # nombre de canaux de sortie
        self.cout = cout
        # nombre de modes en z, y, x
        self.m1, self.m2, self.m3 = modes
        # petite échelle pour des poids initiaux faibles (entraînement stable)
        scale = 1.0 / (cin * cout)
        # 4 blocs de poids complexes (un par « coin » du spectre), taille (cin, cout, 6, 32, 32)
        self.w = nn.ParameterList([
            nn.Parameter(scale * torch.randn(cin, cout, *modes, dtype=torch.cfloat))
            for _ in range(4)])

    # Calcul de la couche
    def forward(self, x):
        # désactive la précision mixte : la FFT se calcule en fp32
        with torch.autocast(device_type=x.device.type, enabled=False):
            # entrée en float32
            x = x.float()
            # dimensions : batch, canaux, profondeur (z), hauteur (y), largeur (x)
            B, _, D, H, W = x.shape
            # FFT 3D réelle sur (z, y, x) : le champ devient une somme d'ondes
            xf = torch.fft.rfftn(x, dim=(-3, -2, -1))
            # spectre de sortie rempli de zéros (les hautes fréquences restent nulles)
            out = torch.zeros(B, self.cout, D, H, W // 2 + 1, dtype=torch.cfloat, device=x.device)
            # nombres de modes gardés
            m1, m2, m3 = self.m1, self.m2, self.m3
            # pour chaque fréquence : multiplie le vecteur des canaux par une matrice apprise
            mul = lambda a, w: torch.einsum("bixyz,ioxyz->boxyz", a, w)
            # coin 1 : basses fréquences positives en z et en y
            out[:, :, :m1, :m2, :m3] = mul(xf[:, :, :m1, :m2, :m3], self.w[0])
            # coin 2 : négatives en z, positives en y
            out[:, :, -m1:, :m2, :m3] = mul(xf[:, :, -m1:, :m2, :m3], self.w[1])
            # coin 3 : positives en z, négatives en y
            out[:, :, :m1, -m2:, :m3] = mul(xf[:, :, :m1, -m2:, :m3], self.w[2])
            # coin 4 : négatives en z et en y
            out[:, :, -m1:, -m2:, :m3] = mul(xf[:, :, -m1:, -m2:, :m3], self.w[3])
            # FFT inverse : retour sur la grille (chaque point dépend de tout le domaine)
            return torch.fft.irfftn(out, s=(D, H, W), dim=(-3, -2, -1))


# Le modèle complet PC-FNO3D
class PCFNO3D(nn.Module):
    # cin=25, width=48, modes=(6,32,32), n_layers=4, pad=(4,8,8), proj_hidden=128, local=branche 3x3x3
    def __init__(self, cin, width, modes, n_layers, pad, proj_hidden=128, local=False):
        # initialise nn.Module
        super().__init__()
        # mémorise le padding
        self.pad = pad
        # Lift : 25 -> 48 -> 48 canaux, convolutions 1x1x1 avec GELU entre les deux
        self.lift = nn.Sequential(nn.Conv3d(cin, width, 1), nn.GELU(), nn.Conv3d(width, width, 1))
        # 4 couches de Fourier
        self.spec = nn.ModuleList([SpectralConv3d(width, width, modes) for _ in range(n_layers)])
        # 4 branches « skip » : convolution 1x1x1 (information ponctuelle)
        self.skip = nn.ModuleList([nn.Conv3d(width, width, 1) for _ in range(n_layers)])
        # Branche locale 3x3x3 : la couche de Fourier ne garde que les grandes echelles (modes tronques),
        # la convolution locale restitue les details fins (vents de pente, contrastes de rugosite).
        # 4 convolutions 3x3x3 (padding=1 garde la même taille), si local=True
        self.local = nn.ModuleList([nn.Conv3d(width, width, 3, padding=1) for _ in range(n_layers)]) \
            if local else None
        # Projection : 48 -> 128 -> 3 canaux (u, v, w)
        self.proj = nn.Sequential(nn.Conv3d(width, proj_hidden, 1), nn.GELU(),
                                  nn.Conv3d(proj_hidden, 3, 1))

    # Calcul du modèle
    def forward(self, x):
        # taille d'origine (12, 250, 500)
        D, H, W = x.shape[-3:]
        # padding en z, y, x
        pz, py, px = self.pad
        # lift : 25 -> 48 canaux
        x = self.lift(x)
        # ajoute des zéros aux bords (évite que la FFT relie les bords opposés)
        x = F.pad(x, (0, px, 0, py, 0, pz))
        # pour chacune des 4 couches :
        for n, (sc, sk) in enumerate(zip(self.spec, self.skip)):
            # branche de Fourier (globale) + branche skip (ponctuelle)
            y = sc(x) + sk(x)
            # + branche locale 3x3x3
            if self.local is not None:
                y = y + self.local[n](x)
            # résultat de la couche
            x = y
            # activation GELU sauf après la dernière couche
            if n < len(self.spec) - 1:
                x = F.gelu(x)
        # retire le padding
        x = x[..., :D, :H, :W]
        # Tete de projection en fp32 : en bf16 la sortie est arrondie a ~3 chiffres (quelques cm/s),
        # ce bruit passe dans les differences centrees et cree un plancher sur le residu de masse (DSO2).
        # projection en pleine précision
        with torch.autocast(device_type=x.device.type, enabled=False):
            # sortie (B, 3, 12, 250, 500)
            return self.proj(x.float())   # (B, 3, NZ, NY, NX) : u, v, w normalises


# Crée le modèle et le place sur le GPU
model = PCFNO3D(C_IN, WIDTH, MODES, N_LAYERS, PAD, PROJ_HIDDEN, local=LOCAL_CONV).to(device)
# Compte les paramètres (un nombre complexe compte pour 2)
n_par = sum(p.numel() * (2 if p.is_complex() else 1) for p in model.parameters())
# Affiche le nombre de paramètres (453 M)
print(f"Parametres : {n_par / 1e6:.2f} M")
# fin du bloc
print("BLOC 7 OK")

## BLOC 8 — Loss physique

In [ ]:
# BLOC 8 — Loss physique
# Dénormalisation : remet la sortie du modèle en m/s
def denorm(pred):
    """(B, 3, ...) normalise -> u, v, w physiques (m/s)."""
    # valeur x écart-type + moyenne, pour u, v, w
    return [pred[:, c] * Y_SD[c] + Y_MU[c] for c in range(3)]

# La loss totale
def compute_loss(pred, y, phys_w):
    """
    pred : (B, 3, NZ, NY, NX) normalise ; y : (B, 4, NZ, NY, NX) physique (u, v, w, rho)
    L_data  : MSE normalisee sur les noeuds valides
    L_mass  : residu de continuite (unites physiques), meme operateur que le generateur
    L_bc    : w = u.grad(z_t) au premier noeud valide de chaque colonne
    L_smooth: gradients horizontaux de u, v (optionnel)
    """
    # prédiction en float32
    pred = pred.float()
    # vérité normalisée comme la prédiction
    y_n = (y[:, :3] - Y_MU[None, :, None, None, None]) / Y_SD[None, :, None, None, None]
    # erreur au carré en chaque point
    err = (pred - y_n) ** 2
    # u, v inchangés ; erreur de w multipliée par le poids de son niveau (jusqu'à x25)
    err = torch.cat([err[:, :2], err[:, 2:3] * W_WEIGHT * W_LEVEL_W[None, None, :, None, None]], dim=1)
    # L_data : moyenne sur les seuls nœuds valides
    l_data = err[:, :, VALID_T].mean()

    # prédiction en m/s
    u, v, w = denorm(pred)
    # vraie densité
    rho = y[:, 3]
    # résidu de continuité de la prédiction
    r = mass_residual(rho, u, v, w)
    # L_mass : moyenne du résidu au carré sur les points intérieurs
    l_mass = (r[:, MASS_MASK_T] ** 2).mean()

    # w attendu au sol : vent horizontal x pente
    w_bc = u * DZT_DX_T + v * DZT_DY_T
    # L_bc : écart au carré (normalisé par sd de w) au premier nœud au-dessus du sol
    l_bc = (((w - w_bc) / Y_SD[2])[:, FIRST_VALID_T] ** 2).mean()

    # Loss sur les gradients horizontaux : erreur relative des variations entre cellules voisines
    # valeur par défaut 0
    l_grad = torch.zeros((), device=pred.device)
    # si le terme est activé :
    if LAMBDA_GRAD > 0:
        # différences entre voisins en x : prédiction et vérité
        gxp, gxt = pred[..., 1:] - pred[..., :-1], y_n[..., 1:] - y_n[..., :-1]
        # différences entre voisins en y
        gyp, gyt = pred[..., 1:, :] - pred[..., :-1, :], y_n[..., 1:, :] - y_n[..., :-1, :]
        # erreur sur ces différences (paires de voisins valides)
        num = ((gxp - gxt) ** 2)[:, :, GRAD_MASK_X_T].sum() + ((gyp - gyt) ** 2)[:, :, GRAD_MASK_Y_T].sum()
        # taille des vraies différences
        den = (gxt ** 2)[:, :, GRAD_MASK_X_T].sum() + (gyt ** 2)[:, :, GRAD_MASK_Y_T].sum() + 1e-12
        # erreur relative : pénalise un champ trop lisse
        l_grad = num / den

    # terme de lissage, 0 par défaut
    l_smooth = torch.zeros((), device=pred.device)
    # (désactivé : LAMBDA_SMOOTH = 0)
    if LAMBDA_SMOOTH > 0:
        # carré des différences entre voisins de u et v
        l_smooth = ((pred[:, :2, :, :, 1:] - pred[:, :2, :, :, :-1]) ** 2).mean() + \
                   ((pred[:, :2, :, 1:, :] - pred[:, :2, :, :-1, :]) ** 2).mean()

    # L = L_data + 0,15 L_grad + phi(t) [L_mass / 5e-7 + 0,1 L_bc]
    total = l_data + LAMBDA_GRAD * l_grad + phys_w * (LAMBDA_MASS * l_mass / MASS_REF + LAMBDA_BC * l_bc
                                                      + LAMBDA_SMOOTH * l_smooth)
    # renvoie la loss totale et chaque terme (pour les logs)
    return total, {"data": l_data.item(), "mass": l_mass.item(), "bc": l_bc.item(),
                   "smooth": l_smooth.item(), "grad": l_grad.item()}

# R2 de u, v, w
def r2_per_component(pred, y):
    """R^2 par composante sur les noeuds valides, en unites physiques."""
    # prédiction en m/s
    u, v, w = denorm(pred.float())
    # liste des résultats
    out = []
    # pour u, v, w :
    for c, p in enumerate((u, v, w)):
        # vérité aux nœuds valides
        t = y[:, c][:, VALID_T]
        # prédiction aux nœuds valides
        q = p[:, VALID_T]
        # erreur du modèle
        ss_res = ((q - t) ** 2).sum()
        # variation de la vérité autour de sa moyenne
        ss_tot = ((t - t.mean()) ** 2).sum()
        # R2 = 1 - erreur / variation
        out.append((1 - ss_res / (ss_tot + 1e-12)).item())
    # renvoie [R2 u, R2 v, R2 w]
    return out

# Rugosité d'un champ (DSO3)
def roughness(a):
    """Rugosite |dA/dx| + |dA/dy| moyennee sur les paires de voisins VALIDES ; a : (B, NZ, NY, NX)."""
    # différences absolues entre voisins en x (paires valides)
    gx = (a[..., :, 1:] - a[..., :, :-1]).abs()[:, GRAD_MASK_X_T]
    # différences absolues entre voisins en y
    gy = (a[..., 1:, :] - a[..., :-1, :]).abs()[:, GRAD_MASK_Y_T]
    # rugosité moyenne
    return gx.mean() + gy.mean()

# fin du bloc
print("BLOC 8 OK")

## BLOC 9 — Entrainement (reprise automatique)

In [ ]:
# BLOC 9 — Entrainement (reprise automatique)
# Optimiseur AdamW : met à jour les poids pour faire baisser la loss (+ weight decay)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# Nombre total de mises à jour : 140 époques x (400 / 2)
total_steps = EPOCHS * math.ceil(STEPS_PER_EPOCH / ACCUM_STEPS)
# Planning OneCycle : le taux monte sur 10 % des pas puis descend en cosinus
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=total_steps,
                                                pct_start=0.1, anneal_strategy="cos")
# Précision mixte : bfloat16 sur GPU, float32 sur CPU
amp_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
# Active la précision mixte seulement sur GPU
use_amp = device.type == "cuda"

# copy : copier un objet (ici le modèle)
import copy
# Copie du modèle pour la moyenne mobile des poids (EMA), en mode évaluation
ema_model = copy.deepcopy(model).eval()
# L'EMA n'est pas entraînée directement : pas de gradients
for _p in ema_model.parameters():
    _p.requires_grad_(False)
# nombre de mises à jour de l'EMA
ema_steps = 0

# Mise à jour de l'EMA après chaque pas d'optimisation
def ema_update():
    """Moyenne exponentielle des poids (decroissance progressive au debut de l'entrainement)."""
    # modifie la variable globale
    global ema_steps
    # un pas de plus
    ema_steps += 1
    # coefficient : 0,999, plus petit au début pour oublier les poids aléatoires initiaux
    d = min(EMA_DECAY, (1 + ema_steps) / (10 + ema_steps))
    # pas de gradient pour ce calcul
    with torch.no_grad():
        # pour chaque poids : ema = d x ema + (1 - d) x poids actuel
        for pe, p in zip(ema_model.parameters(), model.parameters()):
            pe.mul_(d).add_(p.detach(), alpha=1 - d)

# Sauvegardes : last.pt pese plusieurs Go (poids spectraux + EMA + etat de l'optimiseur).
# Le reecrire sur le Drive a chaque epoque (2 min) empeche le Drive de finir l'envoi : apres une
# coupure, le Drive ne contenait qu'une ancienne version. On ecrit donc sur le disque local a chaque
# epoque, et on copie sur le Drive toutes les SAVE_EVERY epoques (le temps que l'envoi se termine).
# copie de fichiers
import shutil
# copie sur le Drive toutes les 5 époques
SAVE_EVERY = 5
# dossier local (disque de Colab)
LOCAL_DIR = (RUN_DIR / "_local") if TEST_MODE else Path("/content/run_local_v7")
# le crée
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
# état complet pour reprendre, réécrit à chaque époque (local)
ckpt_last = LOCAL_DIR / "last.pt"                  # a chaque epoque (disque local)
# meilleur modèle (local)
ckpt_best_local = LOCAL_DIR / "best.pt"
# copie de sécurité de last.pt sur le Drive
drive_last = RUN_DIR / "last.pt"                   # copie de securite sur le Drive
# meilleur modèle sur le Drive (utilisé ensuite par les autres blocs)
ckpt_best = RUN_DIR / "best.pt"                    # utilise par 9b, 10, 10c, 12
# historique des scores
hist_path = RUN_DIR / "history.json"

# Copie local -> Drive
def sync_to_drive():
    """Copie local -> Drive via un fichier temporaire (le fichier du Drive n'est jamais a moitie ecrit)."""
    # chronomètre
    t0 = time.time()
    # pour best.pt et last.pt :
    for src, dst in ((ckpt_best_local, ckpt_best), (ckpt_last, drive_last)):
        # si le fichier local existe
        if src.exists():
            # nom temporaire (ex. best.pt.tmp)
            tmp = dst.with_name(dst.name + ".tmp")
            # copie vers le fichier temporaire
            shutil.copy2(src, tmp)
            # puis renommage (l'ancien fichier reste intact si la copie est coupée)
            os.replace(tmp, dst)
    # durée de la copie
    print(f"      -> sauvegarde copiee sur le Drive ({time.time() - t0:.0f} s)")

# Valeurs de départ : époque 0, meilleur score infini, historique vide
start_epoch, best_val, history = 0, float("inf"), []
# Reprise : la copie locale (meme session Colab) est la plus recente ; sinon celle du Drive
_resume = ckpt_last if ckpt_last.exists() else (drive_last if drive_last.exists() else None)
# S'il y a une sauvegarde :
if _resume is not None:
    # la charge
    ck = torch.load(_resume, map_location=device, weights_only=False)
    # restaure les poids entraînés et l'état de l'optimiseur
    model.load_state_dict(ck.get("model_raw", ck["model"])); optimizer.load_state_dict(ck["optimizer"])
    # restaure l'EMA et son compteur
    ema_model.load_state_dict(ck["model"]); ema_steps = ck.get("ema_steps", 0)
    # restaure le planning du taux d'apprentissage
    scheduler.load_state_dict(ck["scheduler"])
    # reprend à l'époque suivante, avec le meilleur score et l'historique
    start_epoch, best_val, history = ck["epoch"] + 1, ck["best_val"], ck["history"]
    # message
    print(f"Reprise a l'epoque {start_epoch} (meilleure val = {best_val:.5f}) depuis {_resume}")
    # si on reprend depuis le Drive, recopie aussi best.pt en local
    if _resume == drive_last and ckpt_best.exists() and not ckpt_best_local.exists():
        shutil.copy2(ckpt_best, ckpt_best_local)

# Validation d'un réseau
def run_validation(net):
    """Loss de validation et R2 GLOBAL (cumule sur tous les instants), comme au BLOC 10."""
    # mode évaluation
    net.eval()
    # sommes des termes de la loss
    acc = {"data": 0.0, "mass": 0.0, "bc": 0.0, "grad": 0.0}
    # sommes pour le R2 : par composante (erreur², somme t, somme t²)
    stats = np.zeros((3, 3))                       # (composante) x (sse, somme t, somme t^2)
    # nombre total de points
    count = 0
    # pas de gradients
    with torch.no_grad():
        # pour chaque instant de validation :
        for s_raw, sensor, tfeat, y, _ in val_loader:
            # envoi sur le GPU
            s_raw, sensor, tfeat, y = (t.to(device, non_blocking=True) for t in (s_raw, sensor, tfeat, y))
            # précision mixte
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                # prédiction
                pred = net(assemble_input(s_raw, sensor, tfeat))
            # termes de la loss avec la physique à plein poids
            _, parts = compute_loss(pred, y, 1.0)
            # ajoute chaque terme
            for k in acc:
                acc[k] += parts[k]
            # pour u, v, w en m/s :
            for c, p in enumerate(denorm(pred.float())):
                # prédiction et vérité aux nœuds valides, en double précision
                pv, tv = p[:, VALID_T].double(), y[:, c][:, VALID_T].double()
                # ajoute erreur², somme et somme des carrés de la vérité
                stats[c] += [((pv - tv) ** 2).sum().item(), tv.sum().item(), (tv ** 2).sum().item()]
            # ajoute le nombre de points
            count += int(VALID_T.sum().item()) * y.shape[0]
    # nombre d'instants
    n = len(val_loader)
    # R2 global = 1 - erreur / variance totale
    r2 = [1 - stats[c, 0] / (stats[c, 2] - stats[c, 1] ** 2 / count) for c in range(3)]
    # moyennes des termes et R2
    return {k: v / n for k, v in acc.items()}, r2

# Boucle sur les époques (reprend à start_epoch)
for epoch in range(start_epoch, EPOCHS):
    # mode entraînement
    model.train()
    # phi(t) : poids de la physique, 0,1 -> 1 sur les 10 premières époques
    phys_w = min(1.0, (epoch + 1) / max(WARMUP_PHYS_EPOCHS, 1))
    # chronomètre de l'époque
    t0 = time.time()
    # sommes des termes de la loss d'entraînement
    tr = {"total": 0.0, "data": 0.0, "mass": 0.0, "bc": 0.0}
    # remet les gradients à zéro
    optimizer.zero_grad(set_to_none=True)
    # pour chacun des 400 instants de l'époque :
    for step, (s_raw, sensor, tfeat, y, _) in enumerate(train_loader):
        # envoi sur le GPU
        s_raw, sensor, tfeat, y = (t.to(device, non_blocking=True) for t in (s_raw, sensor, tfeat, y))
        # précision mixte bf16
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            # prédiction
            pred = model(assemble_input(s_raw, sensor, tfeat))
        # loss avec le poids physique de l'époque
        loss, parts = compute_loss(pred, y, phys_w)
        # rétropropagation (divisée par 2 car on accumule 2 instants)
        (loss / ACCUM_STEPS).backward()
        # tous les 2 instants (ou au dernier) :
        if (step + 1) % ACCUM_STEPS == 0 or step + 1 == len(train_loader):
            # limite la taille des gradients (stabilité)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # mise à jour des poids
            optimizer.step()
            # remet les gradients à zéro
            optimizer.zero_grad(set_to_none=True)
            # met à jour la moyenne mobile des poids
            ema_update()
            # avance le planning du taux d'apprentissage (sans dépasser la fin)
            if scheduler.last_epoch < total_steps - 1:
                scheduler.step()
        # ajoute la loss totale
        tr["total"] += loss.item()
        # ajoute chaque terme
        for k in ("data", "mass", "bc"):
            tr[k] += parts[k]
    # nombre de pas
    n = len(train_loader)
    # moyennes de l'époque
    tr = {k: v / n for k, v in tr.items()}
    # validation du modèle EMA
    va, r2 = run_validation(ema_model)
    # score de validation = données + masse + condition au sol
    score = va["data"] + LAMBDA_MASS * va["mass"] / MASS_REF + LAMBDA_BC * va["bc"]
    # enregistrement de l'époque
    rec = {"epoch": epoch, "phys_w": phys_w, "lr": optimizer.param_groups[0]["lr"],
           "train": tr, "val": va, "val_r2_u": r2[0], "val_r2_v": r2[1], "val_r2_w": r2[2],
           "val_score": score, "time_s": time.time() - t0}
    # ajouté à l'historique
    history.append(rec)
    # si c'est le meilleur score :
    if score < best_val:
        # nouveau meilleur score
        best_val = score
        # sauvegarde best.pt (poids EMA + configuration de l'architecture)
        torch.save({"model": ema_model.state_dict(), "epoch": epoch, "val_score": score,
                    "config": {"WIDTH": WIDTH, "MODES": MODES, "N_LAYERS": N_LAYERS, "PAD": PAD,
                               "PROJ_HIDDEN": PROJ_HIDDEN, "C_IN": C_IN, "LOCAL_CONV": LOCAL_CONV,
                               "POOL_SURFACE": POOL_SURFACE}}, ckpt_best_local)
    # sauvegarde complète last.pt (pour la reprise) à chaque époque
    torch.save({"model": ema_model.state_dict(), "model_raw": model.state_dict(),
                "optimizer": optimizer.state_dict(), "ema_steps": ema_steps,
                "scheduler": scheduler.state_dict(), "epoch": epoch, "best_val": best_val,
                "history": history}, ckpt_last)
    # écrit l'historique
    with open(hist_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=1)
    # ligne de log de l'époque (* = nouveau meilleur modèle)
    print(f"Ep {epoch + 1:3d}/{EPOCHS} | phys={phys_w:.2f} | train data={tr['data']:.4f} "
          f"mass={tr['mass']:.2e} | val data={va['data']:.4f} mass={va['mass']:.2e} "
          f"bc={va['bc']:.3f} grad={va['grad']:.3f} | R2 u={r2[0]:.3f} v={r2[1]:.3f} w={r2[2]:.3f} "
          f"| {rec['time_s']:.0f} s{' *' if score == best_val else ''}")
    # toutes les 5 époques et à la dernière : copie sur le Drive
    if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
        sync_to_drive()
# À la fin : copie best.pt sur le Drive s'il manque ou si la version locale est plus récente
if not ckpt_best.exists() or (ckpt_best_local.exists()
                              and ckpt_best_local.stat().st_mtime > ckpt_best.stat().st_mtime):
    sync_to_drive()
# fin du bloc
print("BLOC 9 OK")

## BLOC 9b — Raffinement oriente DSO (optionnel, recommande)

In [ ]:
# BLOC 9b — Raffinement oriente DSO (optionnel, recommande)
# Reprend le meilleur modele, renforce progressivement la conservation de masse avec un petit
# taux d'apprentissage, et conserve le checkpoint qui respecte le mieux TOUS les DSO sur la
# validation (et non plus seulement la loss). Score DSO = max des rapports mesure / seuil :
# un score <= 1 signifie que DSO1 (RMSE), DSO2 et DSO3 sont respectes sur la validation.
# (NB : ce bloc a été testé puis NON retenu, car il dégradait les résultats ; seul le BLOC 9 est utilisé)
# nombre d'époques de raffinement
FT_EPOCHS = 20
# petit taux d'apprentissage (ajustements fins)
FT_LR = 1e-4
# référence de la masse pour chaque moitié du raffinement
FT_MASS_REFS = (5e-7, 5e-7)          # masse NON resserree : en v6, la resserrer lissait les champs
                                     # (rugosite 0.82 -> 0.79) sans ameliorer le residu
# poids plus fort du terme de gradient
FT_LAMBDA_GRAD = 0.3                 # priorite a la rugosite (DSO3 : objectif 0.9 - 1.1)
# suffixe des fichiers de cette version
FT_TAG = "v7"                      # fichiers separes : les resultats v6 (best_dso.pt) sont conserves
# point de départ : le meilleur modèle du BLOC 9
ckpt_best = RUN_DIR / "best.pt"      # on repart TOUJOURS du modele de fin d'entrainement
# fichier du modèle retenu selon le score DSO
ckpt_dso = RUN_DIR / f"best_dso_{FT_TAG}.pt"
# sauvegarde de reprise du raffinement (disque local)
ckpt_ft_last = ((RUN_DIR / "_local") if TEST_MODE else Path("/content/run_local_v7")) / f"ft_last_{FT_TAG}.pt"
# crée le dossier
ckpt_ft_last.parent.mkdir(parents=True, exist_ok=True)   # local : plusieurs Go reecrits a chaque epoque
# en mode test :
if TEST_MODE:
    # seulement 2 époques
    FT_EPOCHS = 2

# vrai pour les instants de validation synoptiques (heure pleine multiple de 3)
va_syn = np.array([pd.Timestamp(t).minute == 0 and pd.Timestamp(t).second == 0
                   and pd.Timestamp(t).hour % 3 == 0
                   for t in INDEX.set_index("stem").loc[va_s, "timestamp"]])
# leurs positions
va_syn_idx = np.nonzero(va_syn)[0]
# s'il n'y en a aucun :
if len(va_syn_idx) == 0:
    # on prend tous les instants
    va_syn_idx = np.arange(len(va_s))
# 64 instants répartis régulièrement
va_syn_idx = va_syn_idx[np.unique(np.linspace(0, len(va_syn_idx) - 1,
                                              min(VAL_SAMPLES, len(va_syn_idx))).astype(int))]
# chargeur de validation pour le score DSO (graine 1500)
val_dso_loader = DataLoader(ANEO3D(va_s[va_syn_idx], va_h[va_syn_idx], va_d[va_syn_idx],
                                   train=False, seed=1500), batch_size=1)
# part de nœuds valides de chaque niveau
COVER_ = VALID.reshape(NZ, -1).mean(axis=1)
# niveaux assez couverts pour être évalués
LEV_ = [k for k in range(NZ) if COVER_[k] >= MIN_LEVEL_COVER]

# Calcule les indicateurs des DSO sur la validation
def dso_scores():
    """RMSE globale par niveau (u, v), RMS du residu de masse, rugosite, sur la validation."""
    # mode évaluation
    model.eval()
    # sommes par composante et par niveau : erreur², somme t, somme t²
    sse = np.zeros((3, NZ)); st1 = np.zeros((3, NZ)); st2 = np.zeros((3, NZ))
    # nombre de points par niveau ; listes pour la masse et la rugosité
    n = np.zeros(NZ); mass, rough = [], []
    # pas de gradients
    with torch.no_grad():
        # pour chaque instant :
        for s_raw, sensor, tfeat, y, _ in val_dso_loader:
            # envoi sur le GPU
            s_raw, sensor, tfeat, y = (t.to(device) for t in (s_raw, sensor, tfeat, y))
            # précision mixte
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                # prédiction
                pred = model(assemble_input(s_raw, sensor, tfeat))
            # u, v, w en m/s
            u, v, w = denorm(pred.float())
            # pour chaque niveau évalué :
            for k in LEV_:
                # nœuds valides du niveau
                m = VALID_T[k]
                # pour u, v, w :
                for c, p in enumerate((u, v, w)):
                    # vérité
                    t = y[0, c, k][m].double()
                    # ajoute l'erreur au carré
                    sse[c, k] += ((p[0, k][m].double() - t) ** 2).sum().item()
                    # ajoute somme et somme des carrés de la vérité
                    st1[c, k] += t.sum().item(); st2[c, k] += (t ** 2).sum().item()
                # ajoute le nombre de points
                n[k] += m.sum().item()
            # résidu de continuité
            r = mass_residual(y[:, 3], u, v, w)
            # RMS du résidu (DSO2)
            mass.append(torch.sqrt((r[:, MASS_MASK_T] ** 2).mean()).item())
            # rugosité prédite
            rp = roughness(u) + roughness(v)
            # rugosité vraie
            rt = roughness(y[:, 0]) + roughness(y[:, 1])
            # rapport (DSO3)
            rough.append((rp / rt).item())
    # RMSE par niveau de u et v
    rmse = np.sqrt(sse[:2, LEV_] / n[LEV_])
    # variation totale de la vérité par niveau
    sst = st2[:, LEV_] - st1[:, LEV_] ** 2 / n[LEV_]
    # R2 par niveau
    r2 = 1 - sse[:, LEV_] / np.maximum(sst, 1e-12)
    # résumé : pire RMSE, masse moyenne, rugosité moyenne, pire R2 de u, v, w
    out = {"rmse_max": float(rmse.max()), "mass_rms": float(np.mean(mass)), "rough": float(np.mean(rough)),
           "r2_min_u": float(r2[0].min()), "r2_min_v": float(r2[1].min()), "r2_min_w": float(r2[2].min())}
    # rapports mesure / seuil pour chaque objectif (<= 1 = respecté)
    ratios = [out["rmse_max"] / DSO1_RMSE, out["mass_rms"] / DSO2_RMS,
              max(out["rough"] - 1.0, 0.0) / (DSO3_RANGE[1] - 1.0),   # DSO3 : rugosite <= 1.1
              (1 - float(r2.min())) / (1 - DSO_R2_ALL)]
    # score = le pire rapport
    out["score"] = max(ratios)
    # somme des rapports (pour départager)
    out["score_sum"] = sum(ratios)
    # renvoie le résumé
    return out

# charge le meilleur modèle du BLOC 9
ck = torch.load(ckpt_best, map_location=device, weights_only=False)
# met ses poids dans le modèle
model.load_state_dict(ck["model"])
# nouvel optimiseur avec le petit taux
ft_opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=WEIGHT_DECAY)
# taux qui descend en cosinus jusqu'à 5 % de FT_LR
ft_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    ft_opt, T_max=FT_EPOCHS * math.ceil(STEPS_PER_EPOCH / ACCUM_STEPS), eta_min=FT_LR * 0.05)
# applique le poids de gradient renforcé
LAMBDA_GRAD = FT_LAMBDA_GRAD
# époque de départ et historique
ft_start, ft_hist = 0, []
# scores du modèle de départ
best_dso = dso_scores()
# -1 = avant raffinement
best_dso["epoch"] = -1
# affiche les scores de départ
print(f"Depart : RMSE max={best_dso['rmse_max']:.3f} m/s | masse RMS={best_dso['mass_rms']:.2e} "
      f"| rugosite={best_dso['rough']:.3f} | R2 min u/v/w={best_dso['r2_min_u']:.3f}/"
      f"{best_dso['r2_min_v']:.3f}/{best_dso['r2_min_w']:.3f} | score DSO={best_dso['score']:.3f}")
# s'il existe une sauvegarde de raffinement :
if ckpt_ft_last.exists():
    # la charge
    fk = torch.load(ckpt_ft_last, map_location=device, weights_only=False)
    # restaure poids et optimiseur
    model.load_state_dict(fk["model"]); ft_opt.load_state_dict(fk["optimizer"])
    # restaure le planning et l'époque
    ft_sched.load_state_dict(fk["scheduler"]); ft_start = fk["epoch"] + 1
    # restaure le meilleur score et l'historique
    best_dso, ft_hist = fk["best_dso"], fk["history"]
    # message
    print(f"Reprise du raffinement a l'epoque {ft_start}")
# sinon, si aucun modèle DSO n'existe encore :
elif not ckpt_dso.exists():
    # sauvegarde le modèle de départ comme référence
    torch.save({"model": model.state_dict(), "epoch": ck["epoch"], "val_score": best_dso["score"],
                "dso_val": best_dso}, ckpt_dso)

# boucle de raffinement
for ep in range(ft_start, FT_EPOCHS):
    # référence de masse de cette moitié
    MASS_REF = FT_MASS_REFS[0] if ep < FT_EPOCHS // 2 else FT_MASS_REFS[1]
    # mode entraînement
    model.train()
    # chronomètre
    t0 = time.time()
    # remet les gradients à zéro
    ft_opt.zero_grad(set_to_none=True)
    # pour chaque instant :
    for step, (s_raw, sensor, tfeat, y, _) in enumerate(train_loader):
        # envoi sur le GPU
        s_raw, sensor, tfeat, y = (t.to(device, non_blocking=True) for t in (s_raw, sensor, tfeat, y))
        # précision mixte
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            # prédiction
            pred = model(assemble_input(s_raw, sensor, tfeat))
        # loss avec la physique à plein poids
        loss, parts = compute_loss(pred, y, 1.0)
        # rétropropagation
        (loss / ACCUM_STEPS).backward()
        # tous les 2 instants :
        if (step + 1) % ACCUM_STEPS == 0 or step + 1 == len(train_loader):
            # limite les gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # mise à jour, remise à zéro, avance du planning
            ft_opt.step(); ft_opt.zero_grad(set_to_none=True); ft_sched.step()
    # scores DSO après l'époque
    sc = dso_scores()
    # numéro d'époque
    sc["epoch"] = ep
    # ajouté à l'historique
    ft_hist.append(sc)
    # meilleur si le score est plus bas (puis la somme en cas d'égalité)
    better = (sc["score"], sc["score_sum"]) < (best_dso["score"], best_dso["score_sum"])
    # si meilleur :
    if better:
        # nouveau meilleur
        best_dso = sc
        # sauvegarde le modèle retenu
        torch.save({"model": model.state_dict(), "epoch": ck["epoch"], "val_score": sc["score"],
                    "dso_val": sc, "ft_epoch": ep}, ckpt_dso)
    # sauvegarde de reprise à chaque époque
    torch.save({"model": model.state_dict(), "optimizer": ft_opt.state_dict(),
                "scheduler": ft_sched.state_dict(), "epoch": ep, "best_dso": best_dso,
                "history": ft_hist}, ckpt_ft_last)
    # ligne de log
    print(f"Raffinement {ep + 1:2d}/{FT_EPOCHS} | MASS_REF={MASS_REF:.0e} | RMSE max={sc['rmse_max']:.3f} "
          f"| masse RMS={sc['mass_rms']:.2e} | rugosite={sc['rough']:.3f} | R2 min u/v/w="
          f"{sc['r2_min_u']:.3f}/{sc['r2_min_v']:.3f}/{sc['r2_min_w']:.3f} | score DSO={sc['score']:.3f} "
          f"| {time.time() - t0:.0f} s{' *' if better else ''}")

# les blocs suivants utiliseront le modèle DSO
ckpt_best = ckpt_dso                 # le BLOC 10 evalue le modele retenu sur critere DSO
# bilan
print(f"Modele retenu : score DSO validation = {best_dso['score']:.3f} "
      f"({'DSO 1-3 et R2 >= 0.95 respectes' if best_dso['score'] <= 1 else 'au moins un objectif non respecte'})")
# fin du bloc
print("BLOC 9b OK")

## BLOC 10 — Evaluation sur le test et DSO

In [ ]:
# BLOC 10 — Evaluation sur le test et DSO
# si aucun modèle n'est désigné (ou s'il n'existe pas) :
if "ckpt_best" not in globals() or not Path(ckpt_best).exists():
    # on prend le meilleur modèle du BLOC 9
    ckpt_best = RUN_DIR / "best.pt"
# fichier éventuel du BLOC 9b
_dso_v62 = RUN_DIR / "best_dso_v7.pt"
# s'il existe, il est prioritaire (ici le 9b n'a pas été lancé, donc best.pt est évalué)
if _dso_v62.exists():
    # modèle du raffinement
    ckpt_best = _dso_v62                     # modele retenu par le raffinement v7
# charge le fichier du modèle
ck = torch.load(ckpt_best, map_location=device, weights_only=False)
# met les poids dans le réseau
model.load_state_dict(ck["model"])
# mode évaluation
model.eval()
# dossier de résultats propre à ce modèle
EVAL_DIR = RUN_DIR / f"eval_{Path(ckpt_best).stem}"   # un dossier de resultats par modele evalue
# le crée
EVAL_DIR.mkdir(parents=True, exist_ok=True)
# affiche le modèle évalué
print("Modele evalue :", Path(ckpt_best).name, "| resultats ->", EVAL_DIR)
# affiche son époque et son score de validation
print(f"Meilleur modele : epoque {ck['epoch'] + 1} (score val = {ck['val_score']:.5f})")

# dictionnaire nom d'instant -> horodatage
STAMP_OF = dict(zip(INDEX["stem"], INDEX["timestamp"]))
# part de chaque niveau au-dessus du relief
COVER = VALID.reshape(NZ, -1).mean(axis=1)                 # part du domaine au-dessus du relief
# niveaux évalués (couverture >= 5 %)
EVAL_LEVELS = [k for k in range(NZ) if COVER[k] >= MIN_LEVEL_COVER]

# Évalue le modèle sur un ensemble d'instants
def evaluate(loader, stems):
    """Metriques globales (cumulees sur tous les instants) par niveau et par composante."""
    # pour u, v, w et chaque niveau : [nombre, erreur², somme t, somme t²]
    acc = {c: {k: np.zeros(4) for k in range(NZ)} for c in "uvw"}    # n, sse, sum t, sum t^2
    # somme des vitesses vraies par niveau
    spd_sum = np.zeros(NZ)
    # une ligne de résultats par instant
    per_instant = []
    # pas de gradients
    with torch.no_grad():
        # pour chaque instant :
        for s_raw, sensor, tfeat, y, j in loader:
            # envoi sur le GPU
            s_raw, sensor, tfeat, y = (t.to(device) for t in (s_raw, sensor, tfeat, y))
            # précision mixte
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                # prédiction
                pred = model(assemble_input(s_raw, sensor, tfeat))
            # u, v, w en m/s
            u, v, w = denorm(pred.float())
            # ligne de l'instant : nom et horodatage
            row = {"stem": stems[int(j)], "timestamp": STAMP_OF[stems[int(j)]]}
            # pour chaque composante :
            for c, p in zip("uvw", (u, v, w)):
                # vérité de cette composante
                t = y[:, "uvw".index(c)]
                # pour chaque niveau :
                for k in range(NZ):
                    # nœuds valides
                    m = VALID_T[k]
                    # niveau entièrement sous le relief : ignoré
                    if m.sum() == 0:
                        continue
                    # prédiction et vérité du niveau en double précision
                    pk, tk = p[0, k][m].double(), t[0, k][m].double()
                    # ajoute nombre, erreur², somme, somme des carrés
                    acc[c][k] += np.array([tk.numel(), ((pk - tk) ** 2).sum().item(),
                                           tk.sum().item(), (tk ** 2).sum().item()])
                # tous les nœuds valides du volume
                pv, tv = p[0][VALID_T], t[0][VALID_T]
                # RMSE de cet instant pour la composante
                row[f"rmse_{c}"] = torch.sqrt(((pv - tv) ** 2).mean()).item()
            # pour chaque niveau :
            for k in range(NZ):
                # nœuds valides
                m = VALID_T[k]
                # s'il y en a :
                if m.sum() > 0:
                    # ajoute la vitesse vraie racine(u² + v²)
                    spd_sum[k] += torch.hypot(y[0, 0, k][m], y[0, 1, k][m]).double().sum().item()
            # direction prédite en degrés
            dp = torch.rad2deg(torch.atan2(v[0], u[0]))[VALID_T]
            # direction vraie
            dt_ = torch.rad2deg(torch.atan2(y[0, 1], y[0, 0]))[VALID_T]
            # écart angulaire ramené entre 0 et 360
            dd = torch.abs(dp - dt_) % 360
            # plus petit angle (ex. 350° -> 10°)
            dd = torch.minimum(dd, 360 - dd)
            # vents calmes (< 1 m/s) : direction sans signification
            calm = torch.hypot(y[0, 0], y[0, 1])[VALID_T] < 1.0     # direction non definie par vent calme
            # erreur moyenne de direction hors vents calmes
            row["mae_dir_deg"] = (dd[~calm].mean() if (~calm).any() else dd.mean()).item()
            # résidu de continuité
            r = mass_residual(y[:, 3], u, v, w)
            # RMS du résidu (DSO2)
            row["mass_rms"] = torch.sqrt((r[:, MASS_MASK_T] ** 2).mean()).item()

            # rapport rugosité prédite / vraie (DSO3)
            row["roughness_ratio"] = ((roughness(u) + roughness(v))
                                      / (roughness(y[:, 0]) + roughness(y[:, 1]))).item()
            # ajoute la ligne
            per_instant.append(row)

    # tableau par niveau
    levels = []
    # pour chaque niveau :
    for k in range(NZ):
        # altitude, couverture, vitesse moyenne
        rec = {"z_m": int(Z_LEVELS[k]), "couverture": float(COVER[k]),
               "vitesse_moyenne": float(spd_sum[k] / max(acc["u"][k][0], 1))}
        # pour u, v, w :
        for c in "uvw":
            # récupère les sommes
            n, sse, st, st2 = acc[c][k]
            # aucun point : on passe
            if n == 0:
                continue
            # variation totale de la vérité
            sst = st2 - st ** 2 / n
            # RMSE = racine(erreur² moyenne)
            rec[f"rmse_{c}"] = float(np.sqrt(sse / n))
            # R2 = 1 - erreur / variation
            rec[f"r2_{c}"] = float(1 - sse / sst) if sst > 0 else float("nan")
        # ajoute le niveau
        levels.append(rec)
    # renvoie le tableau par niveau et le tableau par instant
    return pd.DataFrame(levels), pd.DataFrame(per_instant)

# Instant synoptique ?
def is_synoptic(stamp):
    # horodatage
    t = pd.Timestamp(stamp)
    # vrai si heure pleine multiple de 3 (0 h, 3 h, 6 h...)
    return t.minute == 0 and t.second == 0 and t.hour % 3 == 0

# vrai/faux pour chaque instant de test
SYN = np.array([is_synoptic(STAMP_OF[s_]) for s_ in te_s])
# positions des synoptiques et des épisodes rapides
syn_idx, ep_idx = np.nonzero(SYN)[0], np.nonzero(~SYN)[0]

# Chargeur pour une liste d'instants de test
def loader_of(idx, seed):
    # dataset de ces instants (masquage fixé par la graine)
    ds = ANEO3D(te_s[idx], te_h[idx], te_d[idx], train=False, seed=seed)
    # chargeur et noms des instants
    return DataLoader(ds, batch_size=1), te_s[idx]

# évaluation sur les instants synoptiques (sert pour DSO1)
LEVELS, res = evaluate(*loader_of(syn_idx, 2000))                 # DSO1 : instants synoptiques
# évaluation sur les épisodes rapides (test de stress), s'il y en a
LEVELS_EP, res_ep = (evaluate(*loader_of(ep_idx, 2500)) if len(ep_idx)
                     else (pd.DataFrame(), pd.DataFrame()))
# marque le type d'instant
res["type"] = "synoptique"
# si épisodes :
if len(res_ep):
    # marque le type
    res_ep["type"] = "episode"
# enregistre les scores par niveau (synoptique)
LEVELS.to_csv(EVAL_DIR / "test_par_niveau_synoptique.csv", index=False)
# s'il y a des épisodes :
if len(LEVELS_EP):
    # enregistre leurs scores par niveau
    LEVELS_EP.to_csv(EVAL_DIR / "test_par_niveau_episodes.csv", index=False)
# réunit tous les instants
res_all = pd.concat([res, res_ep], ignore_index=True)
# enregistre les scores par instant
res_all.to_csv(EVAL_DIR / "test_par_instant.csv", index=False)

# DSO4 : temps d'inference (assemblage de l'entree inclus)
# prend un instant de test
s_raw, sensor, tfeat, y, _ = next(iter(test_loader))
# l'envoie sur le GPU
s_raw, sensor, tfeat = s_raw.to(device), sensor.to(device), tfeat.to(device)
# pas de gradients
with torch.no_grad():
    # 3 passages d'échauffement (non chronométrés)
    for _ in range(3):
        # prédiction
        model(assemble_input(s_raw, sensor, tfeat))
    # sur GPU :
    if device.type == "cuda":
        # attend la fin des calculs
        torch.cuda.synchronize()
    # liste des durées
    times = []
    # 20 mesures (10 en mode test)
    for _ in range(10 if TEST_MODE else 20):
        # départ du chronomètre
        t0 = time.perf_counter()
        # précision mixte
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            # assemblage de l'entrée + prédiction
            model(assemble_input(s_raw, sensor, tfeat))
        # sur GPU :
        if device.type == "cuda":
            # attend la fin réelle du calcul
            torch.cuda.synchronize()
        # durée de ce passage
        times.append(time.perf_counter() - t0)
# temps moyen d'inférence (s)
t_inf = float(np.mean(times))

# DSO5 et robustesse : fraction de capteurs visibles (sous-ensemble du test)
# résultats de robustesse
robust = []
# 40 instants synoptiques répartis régulièrement
sub = syn_idx[np.unique(np.linspace(0, len(syn_idx) - 1, min(40, len(syn_idx))).astype(int))]
# mémorise la part de capteurs visibles d'origine (50 %)
KEEP_EVAL_SAVE = KEEP_EVAL
# teste 10 %, 25 %, 50 % et 100 % de capteurs visibles
for keep in (0.1, 0.25, 0.5, 1.0):
    # change la part de capteurs visibles
    KEEP_EVAL = keep
    # dataset correspondant
    ds = ANEO3D(te_s[sub], te_h[sub], te_d[sub], train=False, seed=3000)
    # évaluation par niveau
    lv, _ = evaluate(DataLoader(ds, batch_size=1), te_s[sub])
    # garde les niveaux évalués
    lv = lv[lv["couverture"] >= MIN_LEVEL_COVER]
    # pire RMSE et pire R2 de u et v
    robust.append({"capteurs_visibles": keep, "rmse_u_max": lv["rmse_u"].max(),
                   "rmse_v_max": lv["rmse_v"].max(), "r2_u_min": lv["r2_u"].min(),
                   "r2_v_min": lv["r2_v"].min()})
# remet la valeur d'origine
KEEP_EVAL = KEEP_EVAL_SAVE
# en tableau
robust = pd.DataFrame(robust)
# enregistre
robust.to_csv(EVAL_DIR / "robustesse_capteurs.csv", index=False)

# Bilan des DSO
# Texte de conformité
def ok(b):
    # vrai -> CONFORME, faux -> NON CONFORME
    return "CONFORME" if b else "NON CONFORME"

# niveaux évalués seulement
ev = LEVELS[LEVELS["couverture"] >= MIN_LEVEL_COVER]
# DSO1 : RMSE de u et v <= 0,5 m/s à chaque niveau
dso1_rmse = bool((ev[["rmse_u", "rmse_v"]].max(axis=1) <= DSO1_RMSE).all())
# DSO1 : R2 de u et v >= seuil à chaque niveau
dso1_r2 = bool((ev[["r2_u", "r2_v"]].min(axis=1) >= DSO1_R2).all())
# R2 de u, v ET w >= 0,95 à chaque niveau
r2_all = bool((ev[["r2_u", "r2_v", "r2_w"]].min(axis=1) >= DSO_R2_ALL).all())
# DSO2 et DSO3 sur les instants synoptiques, comme DSO1 (les episodes rapides sont un test de stress)
# DSO2 : résidu de masse moyen
mass_rms_mean = float(res["mass_rms"].mean())
# 95e centile du résidu
mass_rms_p95 = float(res["mass_rms"].quantile(0.95))
# résidu maximal
mass_rms_max = float(res["mass_rms"].max())
# DSO3 : rugosité moyenne
rough_mean = float(res["roughness_ratio"].mean())
# résidu moyen, épisodes compris (info)
mass_rms_all = float(res_all["mass_rms"].mean())
# rugosité, épisodes compris (info)
rough_all = float(res_all["roughness_ratio"].mean())
# ligne de robustesse à 25 % de capteurs
r25 = robust[robust["capteurs_visibles"] == 0.25].iloc[0]
# DSO5 : pire RMSE à 25 % <= 0,7 m/s
dso5 = max(r25["rmse_u_max"], r25["rmse_v_max"]) <= DSO5_RMSE

# ligne de séparation
print("=" * 78)
# titre du tableau
print(f"TEST SYNOPTIQUE (DSO1) : {len(res)} instants | capteurs visibles : {KEEP_EVAL:.0%}")
# en-têtes de colonnes
print(" Niveau | couverture | RMSE u  RMSE v  RMSE w |  R2 u    R2 v    R2 w")
# pour chaque niveau :
for _, r in LEVELS.iterrows():
    # mention si le niveau est exclu
    flag = "" if r["couverture"] >= MIN_LEVEL_COVER else "  (exclu : couverture < 5 %)"
    # ligne : altitude, couverture, RMSE et R2 de u, v, w
    print(f" {int(r['z_m']):5d} m |   {r['couverture']:5.1%}   | {r['rmse_u']:.3f}   {r['rmse_v']:.3f}   "
          f"{r['rmse_w']:.3f} | {r['r2_u']:.4f}  {r['r2_v']:.4f}  {r['r2_w']:.4f}{flag}")
# séparation
print("-" * 78)
# verdict DSO1 RMSE
print(f"DSO1  RMSE u, v <= {DSO1_RMSE} m/s a chaque niveau        -> {ok(dso1_rmse)}")
# verdict DSO1 R2 u, v
print(f"      R2 u, v >= {DSO1_R2} a chaque niveau               -> {ok(dso1_r2)}")
# verdict R2 u, v, w
print(f"      R2 u, v, w >= {DSO_R2_ALL} a chaque niveau            -> {ok(r2_all)}")
# RMSE global de u et v : moyenne des RMSE² pondérée par la couverture
_g = {c: float(np.sqrt((ev[f"rmse_{c}"] ** 2 * ev["couverture"]).sum() / ev["couverture"].sum()))
      for c in "uv"}
# vrai si les deux sont <= 0,5
dso1_global = max(_g.values()) <= DSO1_RMSE
# affiche le RMSE global
print(f"      RMSE GLOBAL (tous les noeuds 3D) u = {_g['u']:.3f}  v = {_g['v']:.3f} m/s (seuil {DSO1_RMSE})"
      f" -> {ok(dso1_global)}")
# critère relatif : RMSE <= max(0,5 ; 10 % de la vitesse moyenne) à chaque niveau
omm = bool((ev[["rmse_u", "rmse_v"]].max(axis=1)
             <= np.maximum(0.5, 0.10 * ev["vitesse_moyenne"])).all())
# affiche ce critère (information)
print(f"      (info) critere OMM : RMSE <= max(0.5 m/s ; 10 % de la vitesse moyenne) -> {ok(omm)}")
# verdict DSO2
print(f"DSO2  RMS residu moyen = {mass_rms_mean:.2e} (seuil {DSO2_RMS:.0e}) -> {ok(mass_rms_mean <= DSO2_RMS)}"
      f" | p95 = {mass_rms_p95:.2e} | max = {mass_rms_max:.2e}")
# DSO2 épisodes compris
print(f"      (info) tous instants, episodes compris : RMS residu moyen = {mass_rms_all:.2e}")
# verdict DSO3
print(f"DSO3  rugosite prediction / verite = {rough_mean:.3f} (0.9 - 1.1) -> "
      f"{ok(DSO3_RANGE[0] <= rough_mean <= DSO3_RANGE[1])}")
# DSO3 épisodes compris
print(f"      (info) tous instants, episodes compris : rugosite = {rough_all:.3f}")
# verdict DSO4 (temps en ms)
print(f"DSO4  inference = {t_inf * 1000:.0f} ms (seuil 1 s)              -> {ok(t_inf < DSO4_TIME)}")
# verdict DSO5
print(f"DSO5  RMSE max a 25 % de capteurs = {max(r25['rmse_u_max'], r25['rmse_v_max']):.3f} m/s "
      f"(seuil {DSO5_RMSE}) -> {ok(dso5)}")
# erreur moyenne de direction
print(f"Direction : MAE = {res_all['mae_dir_deg'].mean():.2f} deg (vents >= 1 m/s)")
# séparation
print("=" * 78)
# s'il y a des épisodes rapides :
if len(LEVELS_EP):
    # titre
    print(f"EPISODES RAPIDES (rafales, front, brise) : {len(res_ep)} instants — test de stress, hors DSO1")
    # en-têtes
    print(" Niveau | RMSE u  RMSE v |  R2 u    R2 v")
    # pour chaque niveau évalué :
    for _, r in LEVELS_EP[LEVELS_EP["couverture"] >= MIN_LEVEL_COVER].iterrows():
        # ligne du niveau
        print(f" {int(r['z_m']):5d} m | {r['rmse_u']:.3f}   {r['rmse_v']:.3f} | {r['r2_u']:.4f}  {r['r2_v']:.4f}")
    # séparation
    print("=" * 78)
# titre du détail
print("Detail par type d'instant :")
# pour les synoptiques puis les épisodes :
for _name, _d in (("synoptique", res), ("episodes", res_ep)):
    # s'il y a des instants :
    if len(_d):
        # masse moyenne, p95, rugosité et erreur de direction
        print(f"  {_name:11s} | masse RMS moy = {_d['mass_rms'].mean():.2e} | p95 = {_d['mass_rms'].quantile(0.95):.2e}"
              f" | rugosite = {_d['roughness_ratio'].mean():.3f} | MAE dir = {_d['mae_dir_deg'].mean():.2f} deg")
# séparation
print("=" * 78)
# titre robustesse
print("Robustesse (instants synoptiques repartis sur tous les jours de test) :")
# tableau de robustesse arrondi
print(robust.round(4).to_string(index=False))

# résumé complet de l'évaluation
summary = {"instants_synoptiques": len(res), "instants_episodes": len(res_ep),
           "capteurs_visibles": KEEP_EVAL,
           "par_niveau_synoptique": LEVELS.to_dict(orient="records"),
           "par_niveau_episodes": LEVELS_EP.to_dict(orient="records") if len(LEVELS_EP) else [],
           "dso1_rmse": dso1_rmse, "dso1_rmse_global_u": _g["u"], "dso1_rmse_global_v": _g["v"], "dso1_r2": dso1_r2, "dso2_rms_moyen": mass_rms_mean,
           "dso2_rms_p95": mass_rms_p95, "dso2_rms_max": mass_rms_max,
           "dso2_rms_tous_instants": mass_rms_all, "dso3_rugosite_tous_instants": rough_all,
           "modele": Path(ckpt_best).name,
           "dso3_rugosite": rough_mean, "dso4_inference_s": t_inf,
           "dso5_rmse_25pct": float(max(r25["rmse_u_max"], r25["rmse_v_max"])),
           "mae_direction_deg": float(res_all["mae_dir_deg"].mean())}
# écrit le résumé en JSON
with open(EVAL_DIR / "test_summary.json", "w", encoding="utf-8") as f:
    # contenu
    json.dump(summary, f, indent=2)
# fin du bloc
print("BLOC 10 OK")

## BLOC 10c — Moyenne de predictions (reduction de l'erreur sans reentrainement)

In [ ]:
# BLOC 10c — Reduction de l'erreur sans reentrainement : moyenne de predictions (test-time ensembling)
# A executer APRES les BLOCS 0 a 8 et 10 (utilise evaluate, loader_of, syn_idx, LEVELS).
# Principe : on fait K predictions avec des sous-ensembles aleatoires des capteurs visibles
# (+ la prediction complete) et on les moyenne : les erreurs aleatoires se compensent.
# copy : pour dupliquer le modèle
import copy

# nombre de prédictions moyennées
TTA_K = 8             # nombre de predictions moyennees (8 x ~76 ms = ~0.6 s < 1 s : DSO4 respecte)
# part des capteurs gardée à chaque prédiction
TTA_KEEP = 0.7        # fraction des capteurs visibles gardee a chaque passe
# modèles candidats pour l'ensemble
TTA_CKPTS = [RUN_DIR / "best_dso_v7.pt", RUN_DIR / "best.pt"]
# en mode test :
if TEST_MODE:
    # seulement 2 prédictions
    TTA_K = 2

# Modèle « enveloppe » qui moyenne plusieurs prédictions
class TTAModel(nn.Module):
    # bases : modèles à utiliser ; k : prédictions par modèle ; keep : part de capteurs gardée
    def __init__(self, bases, k=8, keep=0.7, seed=0):
        # initialise nn.Module
        super().__init__()
        # liste des modèles
        self.bases = nn.ModuleList(bases)
        # mémorise les réglages
        self.k, self.keep, self.seed = k, keep, seed

    # Calcul
    def forward(self, x):
        # générateur aléatoire à graine fixe (résultat reproductible)
        g = torch.Generator(device=x.device).manual_seed(self.seed)
        # liste des prédictions
        outs = []
        # pour chaque modèle :
        for base in self.bases:
            # prédiction normale
            outs.append(base(x).float())                       # prediction avec tous les capteurs visibles
            # k - 1 prédictions supplémentaires :
            for _ in range(self.k - 1):
                # carte aléatoire : 1 = capteur gardé (70 %), 0 = retiré
                m = (torch.rand(x.shape[0], 1, 1, x.shape[-2], x.shape[-1], generator=g,
                                device=x.device) < self.keep).to(x.dtype)
                # copie de l'entrée
                xi = x.clone()
                # retire les capteurs non gardés (7 mesures + canal de disponibilité)
                xi[:, :N_SURF + 1] = x[:, :N_SURF + 1] * m    # canaux surface + canal "capteur visible"
                # prédiction avec ces capteurs
                outs.append(base(xi).float())
        # moyenne de toutes les prédictions
        return torch.stack(outs).mean(0)

# modèles chargés
bases = []
# pour chaque fichier candidat :
for p in TTA_CKPTS:
    # s'il existe :
    if p.exists():
        # copie du réseau
        b = copy.deepcopy(model)
        # charge les poids du fichier
        b.load_state_dict(torch.load(p, map_location=device, weights_only=False)["model"])
        # mode évaluation et ajout à la liste
        b.eval(); bases.append(b)
        # message
        print("Modele ajoute a l'ensemble :", p.name)
    # 2 modèles au maximum :
    if len(bases) == 2:          # 2 checkpoints au maximum (temps d'inference)
        # arrêt
        break

# RMSE global de u et v
def global_rmse(levels):
    # niveaux évalués
    ev_ = levels[levels["couverture"] >= MIN_LEVEL_COVER]
    # moyenne des RMSE² pondérée par la couverture, puis racine
    return {c: float(np.sqrt((ev_[f"rmse_{c}"] ** 2 * ev_["couverture"]).sum() / ev_["couverture"].sum()))
            for c in "uv"}

# mémorise le modèle simple
_model_single = model
# résultats de chaque configuration
results = {}
# 3 configurations : 1 modèle seul, 1 modèle x K, tous les modèles x K
for name, kk, bb in (("1 modele, sans TTA", 1, bases[:1]),
                     (f"1 modele, TTA x{TTA_K}", TTA_K, bases[:1]),
                     (f"{len(bases)} modeles, TTA x{TTA_K}", TTA_K, bases)):
    # remplace temporairement « model » (utilisé par evaluate)
    model = TTAModel(bb, k=kk, keep=TTA_KEEP).to(device).eval()
    # chronomètre
    t0 = time.time()
    # évaluation sur les instants synoptiques
    lv, rs = evaluate(*loader_of(syn_idx, 2000))
    # RMSE global
    g = global_rmse(lv)
    # mémorise
    results[name] = (lv, rs, g)
    # affiche RMSE global, pire niveau, masse, rugosité, durée
    print(f"{name:26s} | RMSE global u = {g['u']:.3f}  v = {g['v']:.3f} m/s | "
          f"RMSE max niveau u = {lv[lv['couverture'] >= MIN_LEVEL_COVER]['rmse_u'].max():.3f} | "
          f"masse RMS = {rs['mass_rms'].mean():.2e} | rugosite = {rs['roughness_ratio'].mean():.3f} "
          f"| {time.time() - t0:.0f} s")
# remet le modèle simple
model = _model_single

# Temps d'inference de la meilleure configuration (DSO4)
# configuration avec le plus petit RMSE global de u
best_name = min(results, key=lambda n: results[n][2]["u"])
# son nombre de prédictions
kk = 1 if "sans" in best_name else TTA_K
# ses modèles
bb = bases if best_name.startswith(f"{len(bases)} modeles") else bases[:1]
# construit cette configuration
tta = TTAModel(bb, k=kk, keep=TTA_KEEP).to(device).eval()
# un instant de test
s_raw, sensor, tfeat, y, _ = next(iter(test_loader))
# entrée 25 canaux sur le GPU
xin = assemble_input(s_raw.to(device), sensor.to(device), tfeat.to(device))
# sans gradients, en précision mixte :
with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
    # passage d'échauffement
    tta(xin)
    # sur GPU :
    if device.type == "cuda":
        # attend la fin
        torch.cuda.synchronize()
    # chronomètre puis passage mesuré
    t0 = time.perf_counter(); tta(xin)
    # sur GPU :
    if device.type == "cuda":
        # attend la fin réelle
        torch.cuda.synchronize()
# affiche la meilleure configuration et son temps
print(f"\nMeilleure configuration : {best_name} | inference = {(time.perf_counter() - t0) * 1000:.0f} ms (DSO4 < 1 s)")
# ses scores par niveau
lv = results[best_name][0]
# affiche le tableau par niveau
print(lv[["z_m", "couverture", "rmse_u", "rmse_v", "rmse_w", "r2_u", "r2_v", "r2_w"]].round(4).to_string(index=False))

## BLOC 11 — Figures

In [ ]:
# BLOC 11 — Figures
# matplotlib : bibliothèque de graphiques
import matplotlib
# en mode test (sans écran) :
if TEST_MODE:
    # mode « sans affichage », les figures sont seulement enregistrées
    matplotlib.use("Agg")
# interface de tracé
import matplotlib.pyplot as plt

# dossier des figures
FIG = RUN_DIR / "figures"
# tableau de l'historique : époque, loss data et masse (train / val), R2 de u, v, w
hist = pd.DataFrame([{"epoch": h["epoch"] + 1, "train_data": h["train"]["data"],
                      "val_data": h["val"]["data"], "train_mass": h["train"]["mass"],
                      "val_mass": h["val"]["mass"], "r2_u": h["val_r2_u"], "r2_v": h["val_r2_v"],
                      "r2_w": h["val_r2_w"]} for h in history])

# 1. Courbes d'apprentissage
# figure à 3 graphiques côte à côte
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
# L_data d'entraînement (échelle logarithmique)
ax[0].semilogy(hist["epoch"], hist["train_data"], label="train")
# L_data de validation
ax[0].semilogy(hist["epoch"], hist["val_data"], label="validation")
# titre, axe, légende
ax[0].set_title("L_data (normalisee)"); ax[0].set_xlabel("Epoque"); ax[0].legend()
# L_mass d'entraînement
ax[1].semilogy(hist["epoch"], hist["train_mass"], label="train")
# L_mass de validation
ax[1].semilogy(hist["epoch"], hist["val_mass"], label="validation")
# ligne rouge du seuil DSO2
ax[1].axhline(1e-4, color="red", ls="--", label="DSO2 (1e-4)")
# titre, axe, légende
ax[1].set_title("L_mass (unites physiques)"); ax[1].set_xlabel("Epoque"); ax[1].legend()
# pour u, v, w :
for c in "uvw":
    # courbe du R2 de validation
    ax[2].plot(hist["epoch"], hist[f"r2_{c}"], label=f"R2 {c}")
# ligne rouge du seuil 0,95
ax[2].axhline(0.95, color="red", ls="--", label="DSO1 (0.95)")
# limites de l'axe vertical
ax[2].set_ylim(max(-0.5, hist[["r2_u", "r2_v", "r2_w"]].values.min() - 0.05), 1.0)
# titre, axe, légende
ax[2].set_title("R2 validation"); ax[2].set_xlabel("Epoque"); ax[2].legend()
# mise en page, enregistrement, affichage
fig.tight_layout(); fig.savefig(FIG / "01_apprentissage.png", dpi=150); plt.show()

# 2. R2 et RMSE par niveau (test, metriques globales)
# niveaux évalués
lv = LEVELS[LEVELS["couverture"] >= MIN_LEVEL_COVER]
# figure à 2 graphiques
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
# pour u, v, w avec un marqueur différent :
for c, mk in zip("uvw", "osd"):
    # R2 en fonction de l'altitude
    ax[0].plot(lv[f"r2_{c}"].clip(lower=-1), lv["z_m"], marker=mk, label=c)
    # RMSE en fonction de l'altitude
    ax[1].plot(lv[f"rmse_{c}"], lv["z_m"], marker=mk, label=c)
# seuil R2 et noms des axes
ax[0].axvline(DSO1_R2, color="red", ls="--", label="DSO1"); ax[0].set_xlabel("R2"); ax[0].set_ylabel("Altitude ASL (m)")
# seuil RMSE et nom de l'axe
ax[1].axvline(DSO1_RMSE, color="red", ls="--", label="DSO1"); ax[1].set_xlabel("RMSE (m/s)")
# titres
ax[0].set_title("R2 par niveau (test)"); ax[1].set_title("RMSE par niveau (test)")
# pour chaque graphique :
for a_ in ax:
    # légende et grille
    a_.legend(); a_.grid(alpha=0.3)
# mise en page, enregistrement, affichage
fig.tight_layout(); fig.savefig(FIG / "02_niveaux.png", dpi=150); plt.show()

# 3. Cartes verite / prediction / erreur a 100 m et profil vertical d'une colonne
# premier instant de test
s_raw, sensor, tfeat, y, j = test_ds[0]
# sans gradients :
with torch.no_grad():
    # prédiction ([None] ajoute la dimension batch)
    pred = model(assemble_input(s_raw[None].to(device), sensor[None].to(device), tfeat[None].to(device)))
# u, v, w en m/s, ramenés en numpy
u, v, w = [a[0].float().cpu().numpy() for a in denorm(pred.float())]
# vérité en numpy
yt = y.numpy()
# niveau le plus bas couvrant au moins 90 % du domaine (sinon le relief masque la carte)
k100 = int(next((k for k in range(NZ) if VALID[k].mean() >= 0.9), NZ - 1))
# nœuds valides de ce niveau
vm = VALID[k100]
# vitesse vraie (NaN sous le relief)
spd_t = np.where(vm, np.hypot(yt[0, k100], yt[1, k100]), np.nan)
# vitesse prédite
spd_p = np.where(vm, np.hypot(u[k100], v[k100]), np.nan)
# figure à 3 cartes
fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
# même échelle de couleurs pour vérité et prédiction
lim = (np.nanmin(spd_t), np.nanmax(spd_t))
# pour la vérité puis la prédiction :
for a, data, title in ((ax[0], spd_t, "Verite"), (ax[1], spd_p, "Prediction")):
    # affiche la carte
    im = a.imshow(data, origin="lower", cmap="viridis", vmin=lim[0], vmax=lim[1])
    # titre et barre de couleurs
    a.set_title(f"{title} — vitesse a {int(Z_LEVELS[k100])} m ASL"); plt.colorbar(im, ax=a, label="m/s")
# erreur = prédiction - vérité
err = spd_p - spd_t
# échelle d'erreur : 99e centile (ignore les valeurs extrêmes)
emax = np.nanpercentile(np.abs(err), 99)
# carte d'erreur (rouge = surestimation, bleu = sous-estimation)
im = ax[2].imshow(err, origin="lower", cmap="RdBu_r", vmin=-emax, vmax=emax)
# titre et barre de couleurs
ax[2].set_title("Erreur (prediction - verite)"); plt.colorbar(im, ax=ax[2], label="m/s")
# titre général (date), mise en page, enregistrement, affichage
fig.suptitle(STAMP_OF[te_s[0]]); fig.tight_layout(); fig.savefig(FIG / "03_cartes_vitesse.png", dpi=150); plt.show()

# colonne ayant le plus de niveaux au-dessus du relief
jj, ii = np.unravel_index(np.argmax(VALID.sum(axis=0)), (NY, NX))
# niveaux valides de cette colonne
col = VALID[:, jj, ii]
# figure à 3 graphiques
fig, ax = plt.subplots(1, 3, figsize=(13, 5))
# pour u, v, w :
for a, c, name in zip(ax, range(3), "uvw"):
    # profil vrai (noir)
    a.plot(yt[c, col, jj, ii], Z_LEVELS[col], "k-o", label="verite")
    # profil prédit (rouge pointillé)
    a.plot([u, v, w][c][col, jj, ii], Z_LEVELS[col], "r--s", label="prediction")
    # nom de l'axe et grille
    a.set_xlabel(f"{name} (m/s)"); a.grid(alpha=0.3)
# axe vertical et légende
ax[0].set_ylabel("Altitude ASL (m)"); ax[0].legend()
# titre général
fig.suptitle(f"Profil vertical — colonne (iy={jj}, ix={ii})")
# mise en page, enregistrement, affichage
fig.tight_layout(); fig.savefig(FIG / "04_profil_colonne.png", dpi=150); plt.show()

# 4. Robustesse au nombre de capteurs
# une figure
fig, ax = plt.subplots(figsize=(6, 4.5))
# pire RMSE de u selon le % de capteurs
ax.plot(robust["capteurs_visibles"] * 100, robust["rmse_u_max"], marker="o", label="RMSE u (pire niveau)")
# pire RMSE de v
ax.plot(robust["capteurs_visibles"] * 100, robust["rmse_v_max"], marker="s", label="RMSE v (pire niveau)")
# seuil DSO1
ax.axhline(DSO1_RMSE, color="red", ls="--", label="DSO1 (0.5 m/s)")
# seuil DSO5
ax.axhline(DSO5_RMSE, color="orange", ls="--", label="DSO5 (0.7 m/s)")
# noms des axes, légende, grille
ax.set_xlabel("Capteurs visibles (%)"); ax.set_ylabel("RMSE (m/s)"); ax.legend(); ax.grid(alpha=0.3)
# titre
ax.set_title("Robustesse au nombre de capteurs")
# mise en page, enregistrement, affichage
fig.tight_layout(); fig.savefig(FIG / "05_robustesse.png", dpi=150); plt.show()
# dossier des figures
print("Figures ->", FIG)
# fin du bloc
print("BLOC 11 OK")

## BLOC 12 — Reconstruction 3D d'un instant

In [ ]:
# BLOC 12 — Reconstruction 3D d'un instant
# Deux sources possibles :
#   A) un instant deja en cache (ex. un jour de test) : RECON_STEM = "004_120000"
#   B) un nouveau fichier de mesures de surface (memes colonnes que le dataset) : RECON_CSV = Path(...)
# Sorties (dans RUN_DIR/reconstructions/) :
#   <nom>_3d.npz  : u, v, w, vitesse, direction (NZ, NY, NX), NaN sous le terrain, niveaux, coordonnees
#   <nom>_3d.csv  : un noeud 3D par ligne (point_id, altitude, u, v, w, vitesse, direction)
#   figures       : carte a l'altitude choisie + coupe verticale
# graphiques
import matplotlib.pyplot as plt

# instant du cache à reconstruire
RECON_STEM = None            # ex. "004_120000" ; None = premier instant du test
# ou fichier CSV de nouvelles mesures
RECON_CSV = None             # ex. Path("/content/drive/MyDrive/mesures/surface_2025-06-14_1200.csv")
# part des capteurs utilisés
RECON_KEEP = 1.0             # fraction de capteurs utilises (1.0 = tous les capteurs disponibles)
# altitude de la carte
MAP_ALTITUDE = 120.0         # altitude ASL de la carte (m), ex. hauteur de moyeu d'eolienne
# ligne de la coupe verticale (milieu du domaine)
SECTION_ROW = NY // 2        # ligne (iy) de la coupe verticale

# si le BLOC 9 n'a pas été lancé dans cette session :
if "amp_dtype" not in globals():                       # si le BLOC 9 n'a pas ete execute
    # définit la précision mixte
    amp_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
    # active seulement sur GPU
    use_amp = device.type == "cuda"

# dossier des reconstructions
RECON_DIR = RUN_DIR / "reconstructions"
# le crée
RECON_DIR.mkdir(parents=True, exist_ok=True)

# premier fichier de modèle existant (best_dso_v7 sinon best.pt)
_ckpt_recon = next((p for p in (RUN_DIR / "best_dso_v7.pt", RUN_DIR / "best.pt")
                    if p.exists()))
# le charge
ck = torch.load(_ckpt_recon, map_location=device, weights_only=False)
# affiche le modèle utilisé
print("Modele utilise :", _ckpt_recon.name)
# met les poids dans le réseau
model.load_state_dict(ck["model"])
# mode évaluation
model.eval()

# Lit un CSV de mesures de surface et le met sur la grille
def surface_from_csv(path):
    """Mesures de surface d'un seul instant -> (S (8, NY, NX), heure, jour de l'annee, timestamp)."""
    # colonnes utiles : identifiant, date, présence du capteur, 7 variables
    cols = ["point_id", "timestamp_utc", "mask_i_obs"] + SURF_VARS
    # lecture du CSV
    df = pd.read_csv(path, usecols=cols)
    # garde les points de surface (pas les nœuds 3D)
    df = df[df["point_id"] < NODE_ID_START]
    # date de l'instant
    stamp = pd.Timestamp(df["timestamp_utc"].iloc[0])
    # garde un seul instant
    df = df[df["timestamp_utc"] == df["timestamp_utc"].iloc[0]]
    # tableau vide : 8 canaux x points de la grille fine
    S = np.zeros((8, NY_S * NX_S), dtype=np.float32)
    # position de chaque point sur la grille
    flat = PID_TO_SURF[df["point_id"].values]
    # pour chaque variable :
    for ci, var in enumerate(SURF_VARS):
        # place les valeurs
        S[ci, flat] = df[var].values
    # canal 8 : capteur présent ou non
    S[7, flat] = df["mask_i_obs"].values
    # pooling 2x2 vers la grille du modèle
    S = pool_surface(S)
    # heure décimale
    hour = stamp.hour + stamp.minute / 60 + stamp.second / 3600
    # renvoie surface, heure, jour de l'année, date
    return S, hour, stamp.dayofyear, str(stamp)

# 1. Chargement de l'instant
# vérité inconnue par défaut
Y_true = None
# si un CSV est fourni :
if RECON_CSV is not None:
    # lit les mesures
    S, hour, doy, stamp = surface_from_csv(RECON_CSV)
    # nom des fichiers de sortie d'après la date
    name = f"recon_{pd.Timestamp(stamp).strftime('%Y%m%d_%H%M%S')}"
# sinon (instant du cache) :
else:
    # instant choisi ou premier du test
    stem = RECON_STEM or te_s[0]
    # sa ligne dans l'index
    row = INDEX[INDEX["stem"] == stem].iloc[0]
    # surface
    S = np.load(CACHE_DIR / f"{stem}_S.npy")
    # vérité 3D (pour comparer)
    Y_true = np.load(CACHE_DIR / f"{stem}_Y.npy")
    # heure, jour, date
    hour, doy, stamp = row["hour"], row["doy"], row["timestamp"]
    # nom des fichiers de sortie
    name = f"recon_{stem}"

# générateur aléatoire à graine 0
rng = np.random.default_rng(0)
# capteurs utilisés (tous si RECON_KEEP = 1)
sensor = (S[7] > 0.5) & (rng.random(S[7].shape) < RECON_KEEP)
# 8 variables temps / physique
tfeat = time_geo_features(S, sensor, hour, doy)

# 2. Inference
# surface vers le GPU (avec dimension batch)
s_t = torch.from_numpy(S[:N_SURF])[None].to(device)
# carte des capteurs vers le GPU
m_t = torch.from_numpy(sensor.astype(np.float32))[None].to(device)
# variables temps vers le GPU
t_t = torch.from_numpy(tfeat)[None].to(device)
# départ du chronomètre
t0 = time.perf_counter()
# sans gradients :
with torch.no_grad():
    # précision mixte :
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        # prédiction
        pred = model(assemble_input(s_t, m_t, t_t))
    # u, v, w en m/s, en numpy
    u, v, w = [a[0].float().cpu().numpy() for a in denorm(pred.float())]
# sur GPU :
if device.type == "cuda":
    # attend la fin du calcul
    torch.cuda.synchronize()
# durée de la reconstruction
t_rec = time.perf_counter() - t0

# NaN sous le relief pour u
u = np.where(VALID, u, np.nan)
# pour v
v = np.where(VALID, v, np.nan)
# pour w
w = np.where(VALID, w, np.nan)
# vitesse horizontale
speed = np.hypot(u, v)
# direction d'où vient le vent (0 = nord, 90 = est)
direction = (270.0 - np.degrees(np.arctan2(v, u))) % 360.0     # convention meteo (d'ou vient le vent)
# affiche date, nombre de capteurs et durée
print(f"Instant {stamp} | capteurs utilises : {int(sensor.sum()):,} | reconstruction en {t_rec * 1000:.0f} ms")

# 3. Interpolation a une altitude quelconque (ASL)
# Champ à n'importe quelle altitude entre deux niveaux
def field_at_altitude(field, z):
    """Interpolation lineaire entre niveaux ; NaN sous le terrain ou hors [25, 1000] m."""
    # altitude hors du domaine :
    if z < Z_LEVELS[0] or z > Z_LEVELS[-1]:
        # erreur
        raise ValueError(f"Altitude hors domaine [{Z_LEVELS[0]:.0f}, {Z_LEVELS[-1]:.0f}] m")
    # niveau juste en dessous de z
    k = int(np.clip(np.searchsorted(Z_LEVELS, z) - 1, 0, NZ - 2))
    # position relative de z entre les deux niveaux (0 à 1)
    a = (z - Z_LEVELS[k]) / (Z_LEVELS[k + 1] - Z_LEVELS[k])
    # mélange linéaire des deux niveaux
    out = (1 - a) * field[k] + a * field[k + 1]
    # NaN là où z est sous le relief
    return np.where(z > zt, out, np.nan)

# 4. Sauvegarde
# coordonnées x en mètres
x_m = np.arange(NX) * DX
# coordonnées y en mètres
y_m = np.arange(NY) * DY
# fichier compressé avec champs, niveaux, coordonnées, relief et date
np.savez_compressed(RECON_DIR / f"{name}_3d.npz", u=u.astype(np.float32), v=v.astype(np.float32),
                    w=w.astype(np.float32), speed=speed.astype(np.float32),
                    direction=direction.astype(np.float32), z_levels=Z_LEVELS, x_m=x_m, y_m=y_m,
                    z_terrain=zt.astype(np.float32), timestamp=stamp)

# nœuds 3D du dataset présents sur la grille
node_ok = NODE_FLAT >= 0
# leurs identifiants point_id
node_pid = np.nonzero(node_ok)[0] + NODE_ID_START
# leur position dans le volume
node_flat = NODE_FLAT[node_ok]
# position -> (niveau, ligne, colonne)
kk, jj, ii = np.unravel_index(node_flat, (NZ, NY, NX))
# tableau : une ligne par nœud (identifiant, position, altitudes, u, v, w, vitesse, direction)
out = pd.DataFrame({
    "point_id": node_pid, "timestamp_utc": stamp, "ix": ii * STRIDE, "iy": jj * STRIDE,
    "z_asl_m": Z_LEVELS[kk], "z_agl_m": np.round(Z_LEVELS[kk] - zt[jj, ii], 2),
    "wind_u_m_s": np.round(u[kk, jj, ii], 3), "wind_v_m_s": np.round(v[kk, jj, ii], 3),
    "wind_w_m_s": np.round(w[kk, jj, ii], 4), "wind_speed_m_s": np.round(speed[kk, jj, ii], 3),
    "wind_direction_deg": np.round(direction[kk, jj, ii], 1),
})
# enregistre le CSV
out.to_csv(RECON_DIR / f"{name}_3d.csv", index=False)
# message
print(f"Sauvegarde : {name}_3d.npz et {name}_3d.csv ({len(out):,} noeuds) -> {RECON_DIR}")

# 5. Comparaison a la verite (si l'instant vient du dataset)
# si la vérité est connue :
if Y_true is not None:
    # en-têtes
    print("Niveau (m) |  RMSE u   RMSE v   RMSE w  | R2 vitesse")
    # pour chaque niveau :
    for k in range(NZ):
        # nœuds valides
        m = VALID[k]
        # moins de 10 nœuds : ignoré
        if m.sum() < 10:
            continue
        # RMSE de u, v, w
        e = [np.sqrt(np.mean((p[k][m] - Y_true[c, k][m]) ** 2)) for c, p in enumerate((u, v, w))]
        # vitesse vraie
        st = np.hypot(Y_true[0, k][m], Y_true[1, k][m])
        # R2 de la vitesse
        r2 = 1 - np.sum((speed[k][m] - st) ** 2) / (np.sum((st - st.mean()) ** 2) + 1e-12)
        # ligne du niveau
        print(f"  {Z_LEVELS[k]:7.0f}  | {e[0]:7.3f}  {e[1]:7.3f}  {e[2]:7.4f} | {r2:8.4f}")

# 6. Figures : carte a MAP_ALTITUDE et coupe verticale
# vitesse à 120 m
spd_map = field_at_altitude(speed, MAP_ALTITUDE)
# u à 120 m
u_map = field_at_altitude(u, MAP_ALTITUDE)
# v à 120 m
v_map = field_at_altitude(v, MAP_ALTITUDE)
# figure à 2 graphiques
fig, ax = plt.subplots(1, 2, figsize=(17, 5))
# carte de vitesse en km
im = ax[0].imshow(spd_map, origin="lower", cmap="viridis", extent=[0, NX * DX / 1000, 0, NY * DY / 1000])
# une flèche toutes les « step » cellules (environ 25 en largeur)
step = max(NX // 25, 1)
# positions des flèches en km
Xq, Yq = np.meshgrid(x_m[::step] / 1000, y_m[::step] / 1000)
# flèches blanches du vent (u, v)
ax[0].quiver(Xq, Yq, u_map[::step, ::step], v_map[::step, ::step], color="white", scale=None, width=0.002)
# titre
ax[0].set_title(f"Vent a {MAP_ALTITUDE:.0f} m ASL — {stamp}")
# axes
ax[0].set_xlabel("x (km)"); ax[0].set_ylabel("y (km)")
# barre de couleurs
plt.colorbar(im, ax=ax[0], label="Vitesse (m/s)")

# vitesse dans le plan vertical de la ligne choisie (altitude x distance)
sec = speed[:, SECTION_ROW, :]
# coupe colorée
im = ax[1].pcolormesh(x_m / 1000, Z_LEVELS, sec, cmap="viridis", shading="nearest")
# relief en marron
ax[1].fill_between(x_m / 1000, 0, zt[SECTION_ROW], color="saddlebrown", alpha=0.9, label="relief")
# axe vertical de 0 à 1000 m
ax[1].set_ylim(0, Z_LEVELS[-1])
# titre
ax[1].set_title(f"Coupe verticale (iy = {SECTION_ROW})")
# axes et légende
ax[1].set_xlabel("x (km)"); ax[1].set_ylabel("Altitude ASL (m)"); ax[1].legend(loc="upper right")
# barre de couleurs
plt.colorbar(im, ax=ax[1], label="Vitesse (m/s)")
# mise en page
fig.tight_layout()
# enregistre la figure
fig.savefig(RECON_DIR / f"{name}_carte_coupe.png", dpi=150)
# affiche
plt.show()
# fin du bloc
print("BLOC 12 OK")